In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into DataFrames and keep them in kernel variables for later steps
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Timezone (UTC+8) per rule; include in outputs for clarity
tz = pytz.timezone('Asia/Shanghai')

# Helper to build overview for metric/trace/log
def build_overview(df, name_field):
    total_rows = int(len(df))
    timestamp_min = int(df['timestamp'].min()) if 'timestamp' in df and not df['timestamp'].isnull().all() else None
    timestamp_max = int(df['timestamp'].max()) if 'timestamp' in df and not df['timestamp'].isnull().all() else None
    unique_cmdb_ids = pd.Series(df['cmdb_id'].unique()).astype(str).head(20).tolist() if 'cmdb_id' in df else []
    unique_names = pd.Series(df[name_field].unique()).astype(str).head(20).tolist() if name_field in df else []
    # Top 10 (cmdb_id, name_field) combos
    if ('cmdb_id' in df) and (name_field in df):
        combo_counts = (
            df.groupby(['cmdb_id', name_field])
              .size()
              .reset_index(name='count')
              .sort_values('count', ascending=False)
              .head(10)
        )
        top_combos = combo_counts.to_dict(orient='records')
    else:
        top_combos = []
    overview = {
        'total_rows': total_rows,
        'timestamp_min': timestamp_min,
        'timestamp_max': timestamp_max,
        'timezone': str(tz),
        'unique_cmdb_ids_first20': unique_cmdb_ids,
        f'unique_{name_field}_first20': unique_names,
        'top_10_cmdbid_name_combos': top_combos
    }
    return overview

# Build overviews
overview_metric = build_overview(metric_df, 'kpi_name')
overview_trace = build_overview(trace_df, 'trace_name')
overview_log = build_overview(log_df, 'log_name')

# error_logs special handling: message existence info and sample unique messages (up to 20)
total_rows = int(len(error_logs_df))
timestamp_min = int(error_logs_df['timestamp'].min()) if 'timestamp' in error_logs_df and not error_logs_df['timestamp'].isnull().all() else None
timestamp_max = int(error_logs_df['timestamp'].max()) if 'timestamp' in error_logs_df and not error_logs_df['timestamp'].isnull().all() else None
unique_cmdb_ids_error = pd.Series(error_logs_df['cmdb_id'].unique()).astype(str).head(20).tolist() if 'cmdb_id' in error_logs_df else []
message_exists = 'message' in error_logs_df.columns
sample_messages = pd.Series(error_logs_df['message'].dropna().unique()).astype(str).head(20).tolist() if message_exists else []

overview_error_logs = {
    'total_rows': total_rows,
    'timestamp_min': timestamp_min,
    'timestamp_max': timestamp_max,
    'timezone': str(tz),
    'unique_cmdb_ids_first20': unique_cmdb_ids_error,
    'message_field_exists': message_exists,
    'sample_unique_messages_first20': sample_messages
}

# Keep outputs compact and JSON-serializable by returning dictionaries
overview_metric, overview_trace, overview_log, overview_error_logs

```
Out[1]:
```
summary = (
    "Loaded four telemetry files into kernel variables: metric_df, trace_df, log_df, error_logs_df.\n\n"
    "metric.csv: 319,703 rows (timestamp range 1647747000–1647750600, timezone Asia/Shanghai). "
    "Shows many components (first 20 cmdb_ids include adservice, adservice-0/1/2, cartservice*, checkoutservice*, etc.). "
    "Top (cmdb_id, kpi_name) combos have counts of 61 (examples: shippingservice2-0 with mesh.source... and adservice with app.grpc.* metrics).\n\n"
    "trace.csv: 35,964 rows (timestamp range 1647748800–1647750600). "
    "Top combos include frontend2-0 trace.self.* (31 occurrences) and recommendationservice-1 trace.from_frontend-* (≈30 occurrences).\n\n"
    "log.csv: 1,850 rows (timestamp range 1647748800–1647750600). "
    "Top combos show repeated log.error_count and log.row_count entries for specific pods (many with 31 occurrences, e.g., adservice-2, cartservice-1).\n\n"
    "error_logs.csv: 0 rows (no entries), though a 'message' column exists; no sample messages present.\n\n"
    "All DataFrames remain in memory for follow-up analysis."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

({'total_rows': 319703, 'timestamp_min': 1647747000, 'timestamp_max': 1647750600, 'timezone': 'Asia/Shanghai', 'unique_cmdb_ids_first20': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'], 'unique_kpi_name_first20': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact'], 'top_10_cmdbid_name_combos': [{'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.source.shippingservice2.jaeger-collector.istio_response_bytes.http.202.', 'count': 61}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'count': 61}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'count': 61}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.rr', 'count': 61}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'count': 61}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-6.container_network_receive_MB.eth0', 'count': 61}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-6.container_network_receive_errors.eth0', 'count': 61}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'count': 61}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-6.container_network_receive_packets_dropped.eth0', 'count': 61}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-6.container_network_transmit_MB.eth0', 'count': 61}]}, {'total_rows': 35964, 'timestamp_min': 1647748800, 'timestamp_max': 1647750600, 'timezone': 'Asia/Shanghai', 'unique_cmdb_ids_first20': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0'], 'unique_trace_name_first20': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count'], 'top_10_cmdbid_name_combos': [{'cmdb_id': 'frontend2-0', 'trace_name': 'trace.self.row_count', 'count': 31}, {'cmdb_id': 'frontend2-0', 'trace_name': 'trace.self.error_rate', 'count': 31}, {'cmdb_id': 'frontend2-0', 'trace_name': 'trace.self.duration_p95', 'count': 31}, {'cmdb_id': 'frontend2-0', 'trace_name': 'trace.self.duration_mean', 'count': 31}, {'cmdb_id': 'recommendationservice-1', 'trace_name': 'trace.from_frontend-1.duration_p95', 'count': 30}, {'cmdb_id': 'recommendationservice-1', 'trace_name': 'trace.from_frontend-1.duration_mean', 'count': 30}, {'cmdb_id': 'recommendationservice-1', 'trace_name': 'trace.from_frontend-0.row_count', 'count': 30}, {'cmdb_id': 'recommendationservice-1', 'trace_name': 'trace.from_frontend-0.error_rate', 'count': 30}, {'cmdb_id': 'recommendationservice-1', 'trace_name': 'trace.from_frontend-0.duration_p95', 'count': 30}, {'cmdb_id': 'recommendationservice-1', 'trace_name': 'trace.from_frontend-0.duration_mean', 'count': 30}]}, {'total_rows': 1850, 'timestamp_min': 1647748800, 'timestamp_max': 1647750600, 'timezone': 'Asia/Shanghai', 'unique_cmdb_ids_first20': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1'], 'unique_log_name_first20': ['log.error_count', 'log.row_count'], 'top_10_cmdbid_name_combos': [{'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'adservice-2', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'cartservice-1', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'cartservice-1', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'cartservice-0', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'cartservice-0', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'cartservice-2', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'cartservice-2', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'currencyservice-1', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'currencyservice-1', 'log_name': 'log.error_count', 'count': 31}]}, {'total_rows': 0, 'timestamp_min': None, 'timestamp_max': None, 'timezone': 'Asia/Shanghai', 'unique_cmdb_ids_first20': [], 'message_field_exists': True, 'sample_unique_messages_first20': []})```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing DataFrames: metric_df, trace_df, log_df (assumed loaded in kernel)
# Ensure numeric 'value' columns
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Helper to compute thresholds per (cmdb_id, name_field)
def compute_thresholds(df, name_col):
    groups = df.groupby(['cmdb_id', name_col])
    counts = groups.size().rename('count')
    p95 = groups['value'].quantile(0.95).rename('p95')
    p5 = groups['value'].quantile(0.05).rename('p5')
    thresh = pd.concat([counts, p95, p5], axis=1).reset_index()
    # Round numeric thresholds to 6 decimals
    thresh['p95'] = thresh['p95'].round(6)
    thresh['p5'] = thresh['p5'].round(6)
    # Ensure count is int
    thresh['count'] = thresh['count'].astype(int)
    return thresh

# Compute full thresholds (global using all rows)
metric_thresholds = compute_thresholds(metric_df, 'kpi_name')
trace_thresholds = compute_thresholds(trace_df, 'trace_name')
log_thresholds = compute_thresholds(log_df, 'log_name')

# Build compact top-20 (by count) lists with requested field names: cmdb_id, name_field, count, p95, p5
def top20_compact(thresh_df, original_name_col):
    df = thresh_df.copy()
    # rename the name column generically to 'name_field' for output
    df = df.rename(columns={original_name_col: 'name_field'})
    top20 = df.sort_values('count', ascending=False).head(20)
    # Keep only requested columns and ensure JSON-serializable types
    top20 = top20[['cmdb_id', 'name_field', 'count', 'p95', 'p5']].reset_index(drop=True)
    return top20

top20_metric = top20_compact(metric_thresholds, 'kpi_name')
top20_trace = top20_compact(trace_thresholds, 'trace_name')
top20_log = top20_compact(log_thresholds, 'log_name')

# Display compact results (DataFrames are small: top 20)
metric_thresholds, trace_thresholds, log_thresholds, top20_metric, top20_trace, top20_log

```
Out[2]:
```
summary = (
    "Computed global P95 and P5 per (cmdb_id, name) using all rows and stored full results in kernel variables:\n"
    "- metric_thresholds (5253 groups), trace_thresholds (1232 groups), log_thresholds (62 groups).\n\n"
    "Top highlights (from the top-20 summaries):\n"
    "metric (examples):\n"
    " - shippingservice2-0 / mesh.source... (count=61) -> p95=1940.25, p5=746.25\n"
    " - adservice / app.grpc.count (count=61) -> p95=255.0, p5=200.0\n"
    " - shippingservice2-0 / container network/memory metrics appear frequently with elevated p95 values.\n\n"
    "trace (examples):\n"
    " - frontend2-0 / trace.self.row_count (count=31) -> p95=987.0, p5=792.5\n"
    " - frontend2-0 / trace.self.duration_p95 (count=31) -> p95=0.450604, p5=0.007252\n"
    " - recommendationservice-1 / trace.from_frontend-1.duration_p95 (count=30) -> p95=0.200413, p5=0.002517\n\n"
    "log (examples):\n"
    " - adservice-2 / log.row_count (count=31) -> p95=130.0, p5=106.0\n"
    " - adservice-2 / log.error_count (count=31) -> p95=0.0, p5=0.0 (errors mostly zero)\n"
    " - cartservice-* / log.row_count entries show high p95 values (e.g., >1000) indicating heavy log volumes for some pods.\n\n"
    "Compact top-20 lists are available in kernel variables top20_metric, top20_trace, top20_log for quick inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count           p95            p5
0              adservice                                     app.grpc.count     61    255.000000    200.000000
1              adservice                                       app.grpc.mrt     61      0.002607      0.002412
2              adservice                                        app.grpc.rr     61    100.000000    100.000000
3              adservice                                        app.grpc.sr     61    100.000000    100.000000
4              adservice                                     app.http.count     61     24.000000     24.000000
...                  ...                                                ...    ...           ...           ...
5248  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     61     25.000000      8.500000
5249  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61  63425.000000  24375.000000
5250  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61      0.002050      0.000000
5251  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61     19.000000      8.000000
5252  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61   1940.250000    746.250000

[5253 rows x 5 columns],                  cmdb_id                               trace_name  count        p95         p5
0            adservice-0      trace.from_frontend-0.duration_mean     30   0.000021   0.000016
1            adservice-0       trace.from_frontend-0.duration_p95     30   0.000034   0.000019
2            adservice-0         trace.from_frontend-0.error_rate     30   0.000000   0.000000
3            adservice-0          trace.from_frontend-0.row_count     30  37.550000  22.900000
4            adservice-0      trace.from_frontend-1.duration_mean     30   0.000023   0.000015
...                  ...                                      ...    ...        ...        ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count     30  11.100000   4.000000
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean     30   0.000088   0.000069
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95     30   0.000106   0.000074
1230  shippingservice2-0        trace.from_frontend2-0.error_rate     30   0.000000   0.000000
1231  shippingservice2-0         trace.from_frontend2-0.row_count     30  26.550000   9.450000

[1232 rows x 5 columns],               cmdb_id         log_name  count    p95     p5
0         adservice-0  log.error_count     30    0.0    0.0
1         adservice-0    log.row_count     30  143.1  123.8
2         adservice-1  log.error_count     30    0.0    0.0
3         adservice-1    log.row_count     30  133.1  111.8
4         adservice-2  log.error_count     31    0.0    0.0
..                ...              ...    ...    ...    ...
57  shippingservice-0    log.row_count     31  121.0   89.0
58  shippingservice-1  log.error_count     31    0.0    0.0
59  shippingservice-1    log.row_count     31  120.5   90.0
60  shippingservice-2  log.error_count     31    0.0    0.0
61  shippingservice-2    log.row_count     31  125.0   90.0

[62 rows x 5 columns],                cmdb_id                                         name_field  count           p95            p5
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61  1.940250e+03  7.462500e+02
1            adservice                                     app.grpc.count     61  2.550000e+02  2.000000e+02
2            adservice                                       app.grpc.mrt     61  2.607000e-03  2.412000e-03
3            adservice                                        app.grpc.rr     61  1.000000e+02  1.000000e+02
4            adservice                                        app.grpc.sr     61  1.000000e+02  1.000000e+02
5   shippingservice2-0  container.node-6.container_network_receive_MB....     61  4.586700e-02  3.097700e-02
6   shippingservice2-0  container.node-6.container_network_receive_err...     61  0.000000e+00  0.000000e+00
7   shippingservice2-0  container.node-6.container_network_receive_pac...     61  4.220000e+02  3.060000e+02
8   shippingservice2-0  container.node-6.container_network_receive_pac...     61  0.000000e+00  0.000000e+00
9   shippingservice2-0  container.node-6.container_network_transmit_MB...     61  4.949340e-01  4.528990e-01
10  shippingservice2-0  container.node-6.container_network_transmit_er...     61  0.000000e+00  0.000000e+00
11  shippingservice2-0  container.node-6.container_network_transmit_pa...     61  2.755000e+02  1.920000e+02
12  shippingservice2-0  container.node-6.container_network_transmit_pa...     61  0.000000e+00  0.000000e+00
13  shippingservice2-0  container.node-6.container_memory_failures.hie...     61  0.000000e+00  0.000000e+00
14  shippingservice2-0  container.node-6.container_memory_failures.hie...     61  0.000000e+00  0.000000e+00
15  shippingservice2-0      container.node-6.container_memory_mapped_file     61  1.486848e+06  1.486848e+06
16  shippingservice2-0     container.node-6.container_memory_max_usage_MB     61  1.287266e+02  1.287266e+02
17  shippingservice2-0              container.node-6.container_memory_rss     61  1.110835e+07  1.097318e+07
18  shippingservice2-0             container.node-6.container_memory_swap     61  0.000000e+00  0.000000e+00
19  shippingservice2-0         container.node-6.container_memory_usage_MB     61  3.243750e+01  3.206641e+01,                     cmdb_id                                      name_field  count         p95          p5
0               frontend2-0                            trace.self.row_count     31  987.000000  792.500000
1               frontend2-0                           trace.self.error_rate     31    0.000000    0.000000
2               frontend2-0                         trace.self.duration_p95     31    0.450604    0.007252
3               frontend2-0                        trace.self.duration_mean     31    0.070604    0.004345
4   recommendationservice-1              trace.from_frontend-1.duration_p95     30    0.200413    0.002517
5   recommendationservice-1             trace.from_frontend-1.duration_mean     30    0.119460    0.002348
6   recommendationservice-1                 trace.from_frontend-0.row_count     30   41.000000   33.450000
7   recommendationservice-1                trace.from_frontend-0.error_rate     30    0.000000    0.000000
8   recommendationservice-1              trace.from_frontend-0.duration_p95     30    0.313394    0.002987
9   recommendationservice-1             trace.from_frontend-0.duration_mean     30    0.180918    0.002560
10  recommendationservice-0      trace.to_productcatalogservice-2.row_count     30   26.000000   23.000000
11  recommendationservice-0     trace.to_productcatalogservice-2.error_rate     30    0.000000    0.000000
12               frontend-2            trace.to_cartservice-2.duration_mean     30    0.000180    0.000011
13  recommendationservice-0  trace.to_productcatalogservice-2.duration_mean     30    0.000033    0.000026
14               frontend-2                trace.to_cartservice-2.row_count     30   45.000000   37.450000
15               frontend-2               trace.to_cartservice-2.error_rate     30    0.000000    0.000000
16               frontend-2             trace.to_cartservice-2.duration_p95     30    0.001000    0.000000
17  recommendationservice-0   trace.to_productcatalogservice-2.duration_p95     30    0.000041    0.000029
18               frontend-2  trace.to_productcatalogservice-1.duration_mean     30    0.004821    0.004504
19               frontend-2      trace.to_productcatalogservice-0.row_count     30  238.400000  193.900000,                     cmdb_id       name_field  count     p95     p5
0               adservice-2  log.error_count     31     0.0    0.0
1               adservice-2    log.row_count     31   130.0  106.0
2             cartservice-1    log.row_count     31  1039.5  909.0
3             cartservice-1  log.error_count     31     0.0    0.0
4             cartservice-0    log.row_count     31  1048.5  914.5
5             cartservice-0  log.error_count     31     0.0    0.0
6             cartservice-2  log.error_count     31     0.0    0.0
7             cartservice-2    log.row_count     31  1041.5  899.5
8         currencyservice-1    log.row_count     31   927.0  731.5
9         currencyservice-1  log.error_count     31     0.0    0.0
10        currencyservice-0  log.error_count     31     0.0    0.0
11        currencyservice-0    log.row_count     31   928.0  727.5
12        checkoutservice-2  log.error_count     31     0.0    0.0
13        checkoutservice-2    log.row_count     31    92.0   37.5
14        currencyservice-2    log.row_count     31   912.5  731.5
15        currencyservice-2  log.error_count     31     0.0    0.0
16  recommendationservice-1    log.row_count     31   247.0  206.5
17  recommendationservice-1  log.error_count     31     0.0    0.0
18  recommendationservice-2  log.error_count     31     0.0    0.0
19  recommendationservice-2    log.row_count     31   252.0  212.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse variables assumed in kernel:
# metric_df, trace_df, log_df, metric_thresholds, trace_thresholds, log_thresholds

# 1) compute epoch seconds for the incident window in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 20, 12, 0, 0))
end_dt = tz.localize(datetime(2022, 3, 20, 12, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# 2) filter each dataframe to the incident window (inclusive)
metric_incident_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
trace_incident_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()
log_incident_df = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 3) merge with thresholds (matching by cmdb_id and name field). Keep thresholds columns p95/p5
metric_incident_df = metric_incident_df.merge(
    metric_thresholds[['cmdb_id', 'kpi_name', 'p95', 'p5']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)
trace_incident_df = trace_incident_df.merge(
    trace_thresholds[['cmdb_id', 'trace_name', 'p95', 'p5']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)
log_incident_df = log_incident_df.merge(
    log_thresholds[['cmdb_id', 'log_name', 'p95', 'p5']],
    on=['cmdb_id', 'log_name'],
    how='left'
)

# 4) Aggregation helper to compute requested metrics for metric/trace (both high and low)
def aggregate_metric_trace(df, name_col):
    groups = []
    gb = df.groupby(['cmdb_id', name_col])
    for (cmdb, name), g in gb:
        total_points = int(len(g))
        # thresholds per-group (should be same for all rows in group)
        p95_vals = g['p95'].dropna().unique()
        p5_vals = g['p5'].dropna().unique()
        p95 = float(p95_vals[0]) if len(p95_vals) > 0 else np.nan
        p5 = float(p5_vals[0]) if len(p5_vals) > 0 else np.nan

        # masks for anomalies (comparisons with NaN yield False)
        high_mask = g['value'] >= p95
        low_mask = g['value'] <= p5

        anomaly_count_high = int(high_mask.sum())
        anomaly_count_low = int(low_mask.sum())

        # earliest anomaly timestamp considering both high or low anomalies
        if (high_mask | low_mask).any():
            earliest_anomaly_ts = int(g.loc[high_mask | low_mask, 'timestamp'].min())
        else:
            earliest_anomaly_ts = np.nan

        # severity_high: max(value / p95) among points with value >= p95; set NaN if none or p95 <= 0 to avoid div0
        if anomaly_count_high > 0 and (pd.notna(p95)) and (p95 > 0):
            severity_high = float((g.loc[high_mask, 'value'] / p95).max())
        elif anomaly_count_high > 0 and (pd.notna(p95)) and (p95 == 0):
            # p95 == 0: counting anomalies per rule, but severity is undefined -> set NaN
            severity_high = np.nan
        else:
            severity_high = np.nan

        # severity_low: max(p5 / value) among points with value <= p5 and value>0
        if anomaly_count_low > 0 and (pd.notna(p5)):
            valid = g.loc[low_mask & (g['value'] > 0), 'value']
            if len(valid) > 0:
                severity_low = float((p5 / valid).max())
            else:
                severity_low = np.nan
        else:
            severity_low = np.nan

        groups.append({
            'cmdb_id': cmdb,
            'name_field': name,
            'total_points': total_points,
            'anomaly_count_high': anomaly_count_high,
            'anomaly_count_low': anomaly_count_low,
            'earliest_anomaly_ts': earliest_anomaly_ts,
            'severity_high': severity_high,
            'severity_low': severity_low
        })

    res = pd.DataFrame(groups)
    # numeric rounding to 6 decimals where applicable
    res['severity_high'] = res['severity_high'].round(6)
    res['severity_low'] = res['severity_low'].round(6)
    return res

# 5) Aggregation helper for logs (only high anomalies)
def aggregate_log(df, name_col):
    groups = []
    gb = df.groupby(['cmdb_id', name_col])
    for (cmdb, name), g in gb:
        total_points = int(len(g))
        p95_vals = g['p95'].dropna().unique()
        p95 = float(p95_vals[0]) if len(p95_vals) > 0 else np.nan

        high_mask = g['value'] >= p95
        anomaly_count_high = int(high_mask.sum())

        # for logs we ignore p5 per instruction
        anomaly_count_low = 0

        if high_mask.any():
            earliest_anomaly_ts = int(g.loc[high_mask, 'timestamp'].min())
        else:
            earliest_anomaly_ts = np.nan

        # severity_high: max(value / p95) among points with value >= p95; handle p95 <=0 as NaN for severity
        if anomaly_count_high > 0 and (pd.notna(p95)) and (p95 > 0):
            severity_high = float((g.loc[high_mask, 'value'] / p95).max())
        elif anomaly_count_high > 0 and (pd.notna(p95)) and (p95 == 0):
            severity_high = np.nan
        else:
            severity_high = np.nan

        severity_low = np.nan

        groups.append({
            'cmdb_id': cmdb,
            'name_field': name,
            'total_points': total_points,
            'anomaly_count_high': anomaly_count_high,
            'anomaly_count_low': anomaly_count_low,
            'earliest_anomaly_ts': earliest_anomaly_ts,
            'severity_high': severity_high,
            'severity_low': severity_low
        })

    res = pd.DataFrame(groups)
    res['severity_high'] = res['severity_high'].round(6)
    res['severity_low'] = res['severity_low'].round(6)
    return res

# 6) Run aggregations
metric_incident_report = aggregate_metric_trace(metric_incident_df, 'kpi_name')
trace_incident_report = aggregate_metric_trace(trace_incident_df, 'trace_name')
log_incident_report = aggregate_log(log_incident_df, 'log_name')

# 7) Sorting criteria and pick top 20
metric_incident_report['anomaly_sum'] = metric_incident_report['anomaly_count_high'] + metric_incident_report['anomaly_count_low']
trace_incident_report['anomaly_sum'] = trace_incident_report['anomaly_count_high'] + trace_incident_report['anomaly_count_low']
log_incident_report['anomaly_sum'] = log_incident_report['anomaly_count_high']  # only high for logs

top20_metric_incident = metric_incident_report.sort_values(['anomaly_sum', 'total_points'], ascending=[False, False]).head(20).drop(columns=['anomaly_sum'])
top20_trace_incident = trace_incident_report.sort_values(['anomaly_sum', 'total_points'], ascending=[False, False]).head(20).drop(columns=['anomaly_sum'])
top20_log_incident = log_incident_report.sort_values(['anomaly_sum', 'total_points'], ascending=[False, False]).head(20).drop(columns=['anomaly_sum'])

# Ensure numeric columns have limited precision display (6 decimals already applied where relevant)
# Keep final displayed outputs compact (top 20 as requested)
metric_incident_df, trace_incident_df, log_incident_df, top20_metric_incident, top20_trace_incident, top20_log_incident

```
Out[3]:
```
summary = (
    "Incident window: 2022-03-20 12:00:00 – 12:30:00 (Asia/Shanghai / UTC+8).\n\n"
    "Filtered data sizes: metric_incident_df ~162,473 rows, trace_incident_df 35,964 rows, log_incident_df 1,850 rows.\n\n"
    "Key findings (compact):\n"
    "- Metrics: Many (cmdb_id, kpi) groups show pervasive anomalies across the window (total_points=31 and anomaly_count_high=31 and anomaly_count_low=31 for many groups).\n"
    "  Examples: adservice metrics (app.grpc.rr, app.grpc.sr, app.http.count, multiple JVM/runtime GC metrics) all flagged; severity_high values are often 1.0 (values at or equal to p95) and severity_low often 1.0 where p5==value. Some metrics have NaN severity when thresholds are zero/undefined.\n\n"
    "- Traces: Numerous trace groups (especially error_rate and self.* on frontend2-0 and many trace.from_frontend.* entries) also have all points anomalous in the window (30–31 occurrences). Earliest anomalous timestamps are at or near the window start (1647748800). Severity is often NaN for error_rate (thresholds zero) or small numeric values for duration/row_count when applicable.\n\n"
    "- Logs: Many pods show persistent log.error_count anomalies (30–31 anomalies per group). However, p95 for these error_count series is 0, so severity is undefined (NaN) even though anomaly_count_high == total_points. Some log.row_count groups have large p95 values indicating high volume but error_count anomalies dominate.\n\n"
    "Overall summary: anomalies are widespread across many services/pods during the 12:00–12:30 window, with particularly strong signals from adservice (multiple JVM and RPC KPIs), frontend2-0 traces, and widespread log.error_count hits across pods. Log anomaly severities are not computable where p95==0; many metric anomalies are exactly at the threshold (severity≈1.0). Full aggregated tables are stored in kernel variables for follow-up: metric_incident_df, trace_incident_df, log_incident_df and the compact top-20 summaries in top20_metric_incident, top20_trace_incident, top20_log_incident."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         timestamp             cmdb_id                                           kpi_name         value           p95            p5
0       1647748800           adservice                                     app.grpc.count    240.000000    255.000000    200.000000
1       1647748800           adservice                                       app.grpc.mrt      0.002503      0.002607      0.002412
2       1647748800           adservice                                        app.grpc.rr    100.000000    100.000000    100.000000
3       1647748800           adservice                                        app.grpc.sr    100.000000    100.000000    100.000000
4       1647748800           adservice                                     app.http.count     24.000000     24.000000     24.000000
...            ...                 ...                                                ...           ...           ...           ...
162468  1647750600  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     19.500000     25.000000      8.500000
162469  1647750600  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  57175.000000  63425.000000  24375.000000
162470  1647750600  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.001050      0.002050      0.000000
162471  1647750600  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     17.500000     19.000000      8.000000
162472  1647750600  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1741.250000   1940.250000    746.250000

[162473 rows x 6 columns],         timestamp             cmdb_id                           trace_name      value         p95          p5
0      1647748800         adservice-0  trace.from_frontend-0.duration_mean   0.000019    0.000021    0.000016
1      1647748800         adservice-0   trace.from_frontend-0.duration_p95   0.000027    0.000034    0.000019
2      1647748800         adservice-0     trace.from_frontend-0.error_rate   0.000000    0.000000    0.000000
3      1647748800         adservice-0      trace.from_frontend-0.row_count  29.000000   37.550000   22.900000
4      1647748800         adservice-0  trace.from_frontend-1.duration_mean   0.000016    0.000023    0.000015
...           ...                 ...                                  ...        ...         ...         ...
35959  1647750540  shippingservice2-0     trace.from_frontend2-0.row_count  20.000000   26.550000    9.450000
35960  1647750600         frontend2-0             trace.self.duration_mean   0.005905    0.070604    0.004345
35961  1647750600         frontend2-0              trace.self.duration_p95   0.005905    0.450604    0.007252
35962  1647750600         frontend2-0                trace.self.error_rate   0.000000    0.000000    0.000000
35963  1647750600         frontend2-0                 trace.self.row_count   1.000000  987.000000  792.500000

[35964 rows x 6 columns],        timestamp            cmdb_id         log_name  value    p95     p5
0     1647748800        adservice-0  log.error_count    0.0    0.0    0.0
1     1647748800        adservice-0    log.row_count  138.0  143.1  123.8
2     1647748800        adservice-1  log.error_count    0.0    0.0    0.0
3     1647748800        adservice-1    log.row_count  126.0  133.1  111.8
4     1647748800        adservice-2  log.error_count    0.0    0.0    0.0
...          ...                ...              ...    ...    ...    ...
1845  1647750600  shippingservice-0    log.row_count    2.0  121.0   89.0
1846  1647750600  shippingservice-1  log.error_count    0.0    0.0    0.0
1847  1647750600  shippingservice-1    log.row_count    2.0  120.5   90.0
1848  1647750600  shippingservice-2  log.error_count    0.0    0.0    0.0
1849  1647750600  shippingservice-2    log.row_count    2.0  125.0   90.0

[1850 rows x 6 columns],       cmdb_id                                         name_field  total_points  anomaly_count_high  anomaly_count_low  earliest_anomaly_ts  severity_high  severity_low
2   adservice                                        app.grpc.rr            31                  31                 31         1.647749e+09            1.0           1.0
3   adservice                                        app.grpc.sr            31                  31                 31         1.647749e+09            1.0           1.0
4   adservice                                     app.http.count            31                  31                 31         1.647749e+09            1.0           1.0
6   adservice                                        app.http.rr            31                  31                 31         1.647749e+09            1.0           1.0
7   adservice                                        app.http.sr            31                  31                 31         1.647749e+09            1.0           1.0
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount            31                  31                 31         1.647749e+09            1.0           1.0
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...            31                  31                 31         1.647749e+09            1.0           1.0
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount            31                  31                 31         1.647749e+09            1.0           1.0
11  adservice             runtime.java_lang_ClassLoading_Verbose            31                  31                 31         1.647749e+09            NaN           NaN
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...            31                  31                 31         1.647749e+09            1.0           1.0
13  adservice  runtime.java_lang_Compilation_TotalCompilation...            31                  31                 31         1.647749e+09            1.0           1.0
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...            31                  31                 31         1.647749e+09            1.0           1.0
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...            31                  31                 31         1.647749e+09            1.0           1.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            31                  31                 31         1.647749e+09            1.0           1.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            31                  31                 31         1.647749e+09            1.0           1.0
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            31                  31                 31         1.647749e+09            1.0           1.0
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            31                  31                 31         1.647749e+09            1.0           1.0
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            31                  31                 31         1.647749e+09            1.0           1.0
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            31                  31                 31         1.647749e+09            1.0           1.0
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            31                  31                 31         1.647749e+09            1.0           1.0,            cmdb_id                               name_field  total_points  anomaly_count_high  anomaly_count_low  earliest_anomaly_ts  severity_high  severity_low
850    frontend2-0                    trace.self.error_rate            31                  31                 31           1647748800            NaN           NaN
2      adservice-0         trace.from_frontend-0.error_rate            30                  30                 30           1647748800            NaN           NaN
6      adservice-0         trace.from_frontend-1.error_rate            30                  30                 30           1647748800            NaN           NaN
10     adservice-0         trace.from_frontend-2.error_rate            30                  30                 30           1647748800            NaN           NaN
14     adservice-1         trace.from_frontend-0.error_rate            30                  30                 30           1647748800            NaN           NaN
18     adservice-1         trace.from_frontend-1.error_rate            30                  30                 30           1647748800            NaN           NaN
22     adservice-1         trace.from_frontend-2.error_rate            30                  30                 30           1647748800            NaN           NaN
26     adservice-2         trace.from_frontend-0.error_rate            30                  30                 30           1647748800            NaN           NaN
34     adservice-2         trace.from_frontend-2.error_rate            30                  30                 30           1647748800            NaN           NaN
38    adservice2-0        trace.from_frontend2-0.error_rate            30                  30                 30           1647748800            NaN           NaN
42   cartservice-0  trace.from_checkoutservice-0.error_rate            30                  30                 30           1647748800            NaN           NaN
46   cartservice-0  trace.from_checkoutservice-1.error_rate            30                  30                 30           1647748800            NaN           NaN
50   cartservice-0  trace.from_checkoutservice-2.error_rate            30                  30                 30           1647748800            NaN           NaN
54   cartservice-0         trace.from_frontend-0.error_rate            30                  30                 30           1647748800            NaN           NaN
58   cartservice-0         trace.from_frontend-1.error_rate            30                  30                 30           1647748800            NaN           NaN
62   cartservice-0         trace.from_frontend-2.error_rate            30                  30                 30           1647748800            NaN           NaN
65   cartservice-0                  trace.self.duration_p95            30                  30                 30           1647748800            NaN           NaN
66   cartservice-0                    trace.self.error_rate            30                  30                 30           1647748800            NaN           NaN
70   cartservice-1  trace.from_checkoutservice-0.error_rate            30                  30                 30           1647748800            NaN           NaN
74   cartservice-1  trace.from_checkoutservice-1.error_rate            30                  30                 30           1647748800            NaN           NaN,                     cmdb_id       name_field  total_points  anomaly_count_high  anomaly_count_low  earliest_anomaly_ts  severity_high  severity_low
4               adservice-2  log.error_count            31                  31                  0           1647748800            NaN           NaN
6             cartservice-0  log.error_count            31                  31                  0           1647748800            NaN           NaN
8             cartservice-1  log.error_count            31                  31                  0           1647748800            NaN           NaN
10            cartservice-2  log.error_count            31                  31                  0           1647748800            NaN           NaN
16        checkoutservice-2  log.error_count            31                  31                  0           1647748800            NaN           NaN
18        currencyservice-0  log.error_count            31                  31                  0           1647748800            NaN           NaN
20        currencyservice-1  log.error_count            31                  31                  0           1647748800            NaN           NaN
22        currencyservice-2  log.error_count            31                  31                  0           1647748800            NaN           NaN
30               frontend-0  log.error_count            31                  31                  0           1647748800            NaN           NaN
34               frontend-2  log.error_count            31                  31                  0           1647748800            NaN           NaN
36         paymentservice-0  log.error_count            31                  31                  0           1647748800            NaN           NaN
42  productcatalogservice-0  log.error_count            31                  31                  0           1647748800            NaN           NaN
48  recommendationservice-0  log.error_count            31                  31                  0           1647748800            NaN           NaN
50  recommendationservice-1  log.error_count            31                  31                  0           1647748800            NaN           NaN
52  recommendationservice-2  log.error_count            31                  31                  0           1647748800            NaN           NaN
56        shippingservice-0  log.error_count            31                  31                  0           1647748800            NaN           NaN
58        shippingservice-1  log.error_count            31                  31                  0           1647748800            NaN           NaN
60        shippingservice-2  log.error_count            31                  31                  0           1647748800            NaN           NaN
0               adservice-0  log.error_count            30                  30                  0           1647748800            NaN           NaN
2               adservice-1  log.error_count            30                  30                  0           1647748800            NaN           NaN)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse metric_incident_df from kernel
# Ensure columns exist
assert 'timestamp' in metric_incident_df.columns and 'cmdb_id' in metric_incident_df.columns and 'kpi_name' in metric_incident_df.columns

# Prepare working df
df = metric_incident_df[['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p95', 'p5']].copy()
df = df.sort_values(['cmdb_id', 'kpi_name', 'timestamp']).reset_index(drop=True)

# Determine sampling interval per (cmdb_id, kpi_name) using the mode of positive diffs
def get_sampling_interval(group):
    diffs = group['timestamp'].diff().dropna()
    diffs_pos = diffs[diffs > 0]
    if len(diffs_pos) == 0:
        return np.nan
    # mode; if multiple, take smallest mode
    mode = diffs_pos.mode()
    if len(mode) > 0:
        return int(mode.min())
    else:
        return int(diffs_pos.min())

sampling_map = df.groupby(['cmdb_id', 'kpi_name']).apply(get_sampling_interval).rename('sampling').reset_index()

# Merge sampling back
df = df.merge(sampling_map, on=['cmdb_id', 'kpi_name'], how='left')

# Identify anomalous points (value >= p95 OR value <= p5), handle NaNs safely
df['is_anom'] = False
df.loc[pd.notna(df['p95']) & (df['value'] >= df['p95']), 'is_anom'] = True
df.loc[pd.notna(df['p5']) & (df['value'] <= df['p5']), 'is_anom'] = True

# Compute previous timestamp diff and previous anomaly flag within each group
df['prev_ts'] = df.groupby(['cmdb_id', 'kpi_name'])['timestamp'].shift(1)
df['prev_is_anom'] = df.groupby(['cmdb_id', 'kpi_name'])['is_anom'].shift(1)
df['ts_diff'] = df['timestamp'] - df['prev_ts']

# A new run starts when:
# - current point is anomalous AND (previous point not anomalous OR ts_diff != sampling OR prev_ts is NaN)
df['start_run'] = False
mask_anom = df['is_anom']
cond_prev_not_anom = (~df['prev_is_anom'].fillna(False))
cond_gap = (df['ts_diff'] != df['sampling'])
cond_prev_missing = df['prev_ts'].isna()
df.loc[mask_anom & (cond_prev_not_anom | cond_gap | cond_prev_missing), 'start_run'] = True

# Assign run ids by cumulative sum per group, but only meaningful where anomalous
df['run_id_temp'] = df.groupby(['cmdb_id', 'kpi_name'])['start_run'].cumsum()

# Set run_id only for anomalous rows; non-anomalous rows get NaN
df['run_id'] = df['run_id_temp'].where(df['is_anom'], other=np.nan)

# Aggregate runs: group by cmdb_id, kpi_name, run_id (excluding NaN run_id)
runs = df.dropna(subset=['run_id']).copy()
runs['run_id'] = runs['run_id'].astype(int)

agg = runs.groupby(['cmdb_id', 'kpi_name', 'run_id']).agg(
    fault_start_ts = ('timestamp', 'min'),
    fault_end_ts = ('timestamp', 'max'),
    points = ('timestamp', 'count'),
    max_value = ('value', 'max'),
    min_value = ('value', 'min'),
    p95 = ('p95', 'first'),
    p5 = ('p5', 'first')
).reset_index()

# Compute severities
def compute_max_severity(sub):
    p95 = sub['p95'].iloc[0]
    if pd.isna(p95) or p95 <= 0:
        return np.nan
    return (sub['value'] / p95).max()

def compute_min_severity(sub):
    p5 = sub['p5'].iloc[0]
    vals = sub.loc[sub['value'] > 0, 'value']
    if pd.isna(p5) or len(vals) == 0:
        return np.nan
    return (p5 / vals).max()

# Apply severity calculations by joining aggregated runs with their member rows
# We'll compute per run using groupby on runs DataFrame
sev = runs.groupby(['cmdb_id', 'kpi_name', 'run_id']).apply(
    lambda g: pd.Series({
        'max_severity': compute_max_severity(g),
        'min_severity': compute_min_severity(g),
        'sample_timestamps': g['timestamp'].sort_values().head(3).astype(int).tolist()
    })
).reset_index()

# Merge severities into agg
metric_faults = agg.merge(sev, on=['cmdb_id', 'kpi_name', 'run_id'], how='left')

# earliest_anomaly_ts equals fault_start_ts per requirement
metric_faults['earliest_anomaly_ts'] = metric_faults['fault_start_ts']

# Filter out single-point spikes
metric_faults = metric_faults[metric_faults['points'] > 1].reset_index(drop=True)

# Keep numeric precision to 6 decimals for severities and values
metric_faults['max_severity'] = metric_faults['max_severity'].round(6)
metric_faults['min_severity'] = metric_faults['min_severity'].round(6)
metric_faults['max_value'] = metric_faults['max_value'].round(6)
metric_faults['min_value'] = metric_faults['min_value'].round(6)
metric_faults['p95'] = metric_faults['p95'].round(6)
metric_faults['p5'] = metric_faults['p5'].round(6)

# Select and order columns as requested
metric_faults = metric_faults[[
    'cmdb_id', 'kpi_name', 'fault_start_ts', 'fault_end_ts', 'points',
    'max_value', 'min_value', 'p95', 'p5', 'max_severity', 'min_severity',
    'earliest_anomaly_ts', 'sample_timestamps'
]]

# For sorting, treat NaN max_severity as -inf so they go to bottom when sorting desc
metric_faults['_sort_max_sev'] = metric_faults['max_severity'].fillna(-np.inf)
metric_faults['_sort_points'] = metric_faults['points']

top20_metric_faults = metric_faults.sort_values(['_sort_max_sev', '_sort_points'], ascending=[False, False]).head(20).drop(columns=['_sort_max_sev', '_sort_points'])

# Ensure sample_timestamps are lists of ints (JSON-serializable)
top20_metric_faults['sample_timestamps'] = top20_metric_faults['sample_timestamps'].apply(lambda x: [int(t) for t in x])

# Store metric_faults in kernel variable (already assigned) and display compact top20
metric_faults, top20_metric_faults

```
Out[4]:
```
summary_lines = []
total_faults = int(metric_faults.shape[0])
summary_lines.append(f"Metric fault detection summary (faults longer than 1 point): total_faults = {total_faults}.")
summary_lines.append("Top faults (sorted by max_severity desc, then points): showing up to 10 examples with key fields.")
cols = ['cmdb_id', 'kpi_name', 'points', 'max_severity', 'min_severity', 'fault_start_ts', 'fault_end_ts', 'sample_timestamps']
for i, row in top20_metric_faults.head(10).iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    points = int(row['points'])
    max_sev = row['max_severity'] if not pd.isna(row['max_severity']) else "NaN"
    min_sev = row['min_severity'] if not pd.isna(row['min_severity']) else "NaN"
    start_ts = int(row['fault_start_ts'])
    end_ts = int(row['fault_end_ts'])
    samples = row['sample_timestamps']
    summary_lines.append(
        f"- {cmdb} | {kpi} | points={points} | max_severity={max_sev} | min_severity={min_sev} | start={start_ts} | end={end_ts} | sample_ts={samples}"
    )

summary_lines.append(
    "Observations: top max_severity faults are memory/failure and I/O related (e.g., recommendationservice container memory failures with very high max_severity values; node-1 I/O and TCP retransmission also present). Single-point spikes were removed."
)

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  fault_start_ts  fault_end_ts  points  max_value  min_value        p95     p5  max_severity  min_severity  earliest_anomaly_ts                     sample_timestamps  _sort_max_sev  _sort_points
0              adservice                                        app.grpc.rr      1647748800    1647750600      31  100.00000      100.0  100.00000  100.0          1.00      1.000000           1647748800  [1647748800, 1647748860, 1647748920]           1.00            31
1              adservice                                        app.grpc.sr      1647748800    1647750600      31  100.00000      100.0  100.00000  100.0          1.00      1.000000           1647748800  [1647748800, 1647748860, 1647748920]           1.00            31
2              adservice                                     app.http.count      1647748800    1647750600      31   24.00000       24.0   24.00000   24.0          1.00      1.000000           1647748800  [1647748800, 1647748860, 1647748920]           1.00            31
3              adservice                                        app.http.rr      1647748800    1647750600      31  100.00000      100.0  100.00000  100.0          1.00      1.000000           1647748800  [1647748800, 1647748860, 1647748920]           1.00            31
4              adservice                                        app.http.sr      1647748800    1647750600      31  100.00000      100.0  100.00000  100.0          1.00      1.000000           1647748800  [1647748800, 1647748860, 1647748920]           1.00            31
...                  ...                                                ...             ...           ...     ...        ...        ...        ...    ...           ...           ...                  ...                                   ...            ...           ...
5127  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...      1647750120    1647750240       3   26.50000        8.5   25.00000    8.5          1.06      1.000000           1647750120  [1647750120, 1647750180, 1647750240]           1.06             3
5128  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      1647749220    1647749400       4    0.00205        0.0    0.00205    0.0          1.00      0.000000           1647749220  [1647749220, 1647749280, 1647749340]           1.00             4
5129  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      1647749580    1647750180      11    0.00000        0.0    0.00205    0.0          0.00           NaN           1647749580  [1647749580, 1647749640, 1647749700]           0.00            11
5130  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      1647750480    1647750540       2    0.00000        0.0    0.00205    0.0          0.00           NaN           1647750480              [1647750480, 1647750540]           0.00             2
5131  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      1647750180    1647750240       2   19.00000        7.5   19.00000    8.0          1.00      1.066667           1647750180              [1647750180, 1647750240]           1.00             2

[5132 rows x 15 columns],                        cmdb_id                                           kpi_name  fault_start_ts  fault_end_ts  points     max_value  min_value            p95    p5  max_severity  min_severity  earliest_anomaly_ts                     sample_timestamps
4511   recommendationservice-1  container.node-6.container_memory_failures.con...      1647748800    1647749520      13  1.329900e+04       0.00      49.500000  0.00    268.666667      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
4514   recommendationservice-1  container.node-6.container_memory_failures.hie...      1647748800    1647749520      13  1.329900e+04       0.00      49.500000  0.00    268.666667      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
4611   recommendationservice-2  container.node-6.container_memory_failures.con...      1647748800    1647749520      13  1.275450e+04       0.00      66.000000  0.00    193.250000      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
4614   recommendationservice-2  container.node-6.container_memory_failures.hie...      1647748800    1647749520      13  1.275450e+04       0.00      66.000000  0.00    193.250000      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
4512   recommendationservice-1  container.node-6.container_memory_failures.con...      1647749640    1647750600      17  8.035500e+03       0.00      49.500000  0.00    162.333333      0.000000           1647749640  [1647749640, 1647749700, 1647749760]
4515   recommendationservice-1  container.node-6.container_memory_failures.hie...      1647749640    1647750600      17  8.035500e+03       0.00      49.500000  0.00    162.333333      0.000000           1647749640  [1647749640, 1647749700, 1647749760]
4411   recommendationservice-0  container.node-6.container_memory_failures.con...      1647749700    1647750600      16  1.552650e+04       0.00      99.000000  0.00    156.833333      0.000000           1647749700  [1647749700, 1647749760, 1647749820]
4414   recommendationservice-0  container.node-6.container_memory_failures.hie...      1647749700    1647750600      16  1.552650e+04       0.00      99.000000  0.00    156.833333      0.000000           1647749700  [1647749700, 1647749760, 1647749820]
4612   recommendationservice-2  container.node-6.container_memory_failures.con...      1647749760    1647750600      15  8.712000e+03       0.00      66.000000  0.00    132.000000      0.000000           1647749760  [1647749760, 1647749820, 1647749880]
4615   recommendationservice-2  container.node-6.container_memory_failures.hie...      1647749760    1647750600      15  8.712000e+03       0.00      66.000000  0.00    132.000000      0.000000           1647749760  [1647749760, 1647749820, 1647749880]
4410   recommendationservice-0  container.node-6.container_memory_failures.con...      1647748800    1647749580      14  1.197900e+04       0.00      99.000000  0.00    121.000000      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
4413   recommendationservice-0  container.node-6.container_memory_failures.hie...      1647748800    1647749580      14  1.197900e+04       0.00      99.000000  0.00    121.000000      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
3184                    node-1                                    system.io.rkb_s      1647748800    1647750540      30  1.249300e+05       0.00    1806.000000  0.00     69.174972      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
3198                    node-1                        system.net.tcp.retrans_segs      1647748920    1647749340       8  7.050000e+00       0.00       0.120000  0.00     58.750000      0.000000           1647748920  [1647748920, 1647748980, 1647749040]
4710  recommendationservice2-0      container.node-6.container_memory_mapped_file      1647748800    1647750600      31  6.217728e+06       0.00  135168.000000  0.00     46.000000      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
3181                    node-1                                  system.io.r_await      1647748800    1647750360      27  1.586000e+01       0.00       0.670000  0.00     23.671642      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
3186                    node-1                                     system.io.util      1647749280    1647749340       2  4.875000e+01      47.60       3.900000  0.25     12.500000      0.005252           1647749280              [1647749280, 1647749340]
2136         currencyservice-2  container.node-6.container_cpu_cfs_throttled_s...      1647748800    1647750600      31  5.981600e-02       0.00       0.004809  0.00     12.438296      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
3177                    node-1                                 system.io.avg_q_sz      1647748800    1647749820      18  1.106000e+01       0.00       0.910000  0.00     12.153846      0.000000           1647748800  [1647748800, 1647748860, 1647748920]
3180                    node-1                                    system.io.await      1647749280    1647749340       2  1.541000e+01      14.38       1.930000  0.22      7.984456      0.015299           1647749280              [1647749280, 1647749340])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse metric_faults from kernel (assumed present)
df = metric_faults.copy()

# Ensure required columns exist
required_cols = {'cmdb_id', 'kpi_name', 'fault_start_ts', 'points', 'max_value', 'p95', 'max_severity', 'min_value', 'p5', 'min_severity'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"metric_faults is missing required columns: {missing}")

# 1) Top 20 faults sorted by fault_start_ts ascending
top20_faults_by_start = (
    df.sort_values('fault_start_ts', ascending=True)
      .loc[:, ['cmdb_id', 'kpi_name', 'fault_start_ts', 'points', 'max_value', 'p95', 'max_severity', 'min_value', 'p5', 'min_severity']]
      .copy()
)

# Round numeric columns to 6 decimals where applicable
for col in ['max_value', 'p95', 'max_severity', 'min_value', 'p5', 'min_severity']:
    if col in top20_faults_by_start.columns:
        top20_faults_by_start[col] = top20_faults_by_start[col].apply(lambda x: round(float(x), 6) if pd.notna(x) else np.nan)

top20_faults_by_start = top20_faults_by_start.head(20).reset_index(drop=True)

# 2) Fault-start-time clusters aggregation
# Build example_cmdb_ids (first-occurrence order) per fault_start_ts
grouped = df.groupby('fault_start_ts', sort=True)
example_cmdbs = grouped['cmdb_id'].apply(lambda s: list(pd.Series(s.values).astype(str).drop_duplicates().tolist()[:10])).reset_index(name='example_cmdb_ids')

clusters = (
    grouped.agg(
        fault_count=('fault_start_ts', 'size'),
        distinct_cmdb_count=('cmdb_id', 'nunique')
    ).reset_index()
)

fault_start_clusters = clusters.merge(example_cmdbs, on='fault_start_ts')
# Keep only top 20 by ascending timestamp
fault_start_clusters = fault_start_clusters.sort_values('fault_start_ts', ascending=True).head(20).reset_index(drop=True)

# Ensure JSON-serializable types (ints and lists)
fault_start_clusters['fault_count'] = fault_start_clusters['fault_count'].astype(int)
fault_start_clusters['distinct_cmdb_count'] = fault_start_clusters['distinct_cmdb_count'].astype(int)
fault_start_clusters['example_cmdb_ids'] = fault_start_clusters['example_cmdb_ids'].apply(lambda lst: [str(x) for x in lst])

# Store results in kernel variables and display compactly
top20_faults_by_start, fault_start_clusters

```
Out[5]:
```
summary = (
    "Results from metric_faults:\n\n"
    "1) Top-20 faults by start time (earliest first):\n"
    "- Many of the earliest faults (starting at 1647748800) are mesh/container metrics on frontend/frontend2 with value==0 and p95==0, producing NaN severity values. Several other faults later include non-zero I/O and memory metrics.\n"
    "- The top-20 table (stored in kernel as top20_faults_by_start) includes for each fault: cmdb_id, kpi_name, fault_start_ts, points, max_value, p95, max_severity, min_value, p5, min_severity (numeric fields rounded to 6 decimals).\n\n"
    "2) Fault-start-time clusters:\n"
    "- There is a very large cluster at fault_start_ts = 1647748800 with fault_count = 3711 affecting 59 distinct cmdb_id values (example cmdb_ids provided).\n"
    "- Subsequent timestamps (1647748860, 1647748920, 1647748980, ...) show many smaller clusters (tens of faults each) indicating many faults starting in close succession across many components.\n"
    "- The cluster table is stored in kernel as fault_start_clusters and lists fault_start_ts, fault_count, distinct_cmdb_count, and example_cmdb_ids (up to 10).\n\n"
    "Interpretation:\n"
    "- Faults are highly concentrated at the incident window start (12:00:00 Asia/Shanghai) and remain widespread across many services/nodes. Many zero-valued mesh/container metrics produce undefined severities (p95==0). Other high-severity faults include container memory failures and node I/O/TCP retransmission metrics seen elsewhere in the faults table.\n\n"
    "Stored variables for next steps: top20_faults_by_start, fault_start_clusters, and metric_faults (full faults table)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name  fault_start_ts  points  max_value  p95  max_severity  min_value   p5  min_severity
0    frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
1    frontend-2  mesh.source.frontend.unknown.istio_request_mes...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
2    frontend-2  mesh.source.frontend.unknown.istio_request_dur...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
3    frontend-2  mesh.source.frontend.unknown.istio_request_dur...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
4    frontend-2  mesh.source.frontend.unknown.istio_request_dur...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
5    frontend-2  mesh.source.frontend.unknown.istio_request_byt...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
6    frontend-2  mesh.source.frontend.unknown.istio_request_byt...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
7    frontend-2  mesh.source.frontend.unknown.istio_request_byt...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
8    frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
9    frontend-2  mesh.source.frontend.unknown.istio_requests.gr...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
10   frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
11   frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
12   frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
13   frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
14   frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
15   frontend-2  mesh.source.frontend.recommendationservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
16   frontend-2  mesh.source.frontend.productcatalogservice.ist...      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
17  frontend2-0     container.node-6.container_fs_inodes./dev/vda1      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
18  frontend2-0      container.node-6.container_fs_reads./dev/vda1      1647748800      31        0.0  0.0           NaN        0.0  0.0           NaN
19  frontend2-0       container.node-6.container_fs_reads./dev/vda      1647748800      31        6.0  0.0           NaN        0.0  0.0           0.0,     fault_start_ts  fault_count  distinct_cmdb_count                                   example_cmdb_ids
0       1647748800         3711                   59  [adservice, adservice-0, adservice-1, adservic...
1       1647748860           21                    8  [checkoutservice2-0, node-2, node-3, node-4, n...
2       1647748920           46                   24  [adservice, adservice-0, cartservice, cartserv...
3       1647748980           53                   28  [adservice-2, cartservice-0, checkoutservice-0...
4       1647749040           33                   15  [adservice2-0, cartservice-2, checkoutservice-...
5       1647749100           16                   12  [cartservice2-0, emailservice, emailservice-1,...
6       1647749160           24                   15  [adservice, adservice-0, cartservice-2, checko...
7       1647749220           47                   16  [adservice, adservice-0, cartservice-0, checko...
8       1647749280           25                   13  [adservice-1, adservice-2, cartservice2-0, cur...
9       1647749340           53                   12  [adservice-2, cartservice2-0, checkoutservice-...
10      1647749400           22                   12  [cartservice-1, checkoutservice-0, checkoutser...
11      1647749460           29                   14  [adservice, adservice2, cartservice-0, cartser...
12      1647749520           66                   20  [adservice, adservice-1, adservice2-0, checkou...
13      1647749580           32                   19  [adservice-1, adservice2, adservice2-0, checko...
14      1647749640           31                   17  [adservice-1, adservice-2, adservice2, cartser...
15      1647749700           53                   20  [adservice, cartservice-1, cartservice2-0, che...
16      1647749760           19                   15  [adservice, cartservice-1, checkoutservice-1, ...
17      1647749820           38                   20  [adservice, adservice-0, adservice2, adservice...
18      1647749880           39                   17  [adservice, checkoutservice-0, emailservice-0,...
19      1647749940           26                   17  [adservice-0, adservice2-0, cartservice-2, car...)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse kernel variables: fault_start_clusters, metric_faults

# 1) Top 10 fault_start_ts by fault_count DESC
top_clusters_by_count = (
    fault_start_clusters
    .sort_values('fault_count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Ensure JSON-serializable types
top_clusters_by_count['fault_start_ts'] = top_clusters_by_count['fault_start_ts'].astype(int)
top_clusters_by_count['fault_count'] = top_clusters_by_count['fault_count'].astype(int)
top_clusters_by_count['distinct_cmdb_count'] = top_clusters_by_count['distinct_cmdb_count'].astype(int)
top_clusters_by_count['example_cmdb_ids'] = top_clusters_by_count['example_cmdb_ids'].apply(lambda lst: [str(x) for x in lst])

# 2) For the top 2 clusters by fault_count, aggregate top 10 cmdb_id candidates per cluster
top2_ts = top_clusters_by_count['fault_start_ts'].head(2).tolist()

candidates_rows = []
for ts in top2_ts:
    subset = metric_faults[metric_faults['fault_start_ts'] == ts].copy()
    if subset.shape[0] == 0:
        continue
    # Ensure numeric max_severity
    subset['max_severity'] = pd.to_numeric(subset['max_severity'], errors='coerce')
    # Group by cmdb_id
    gb = subset.groupby('cmdb_id')
    agg = gb.agg(
        faults_in_cluster = ('cmdb_id', 'size'),
        distinct_kpi_count = ('kpi_name', lambda s: int(s.nunique())),
        max_max_severity = ('max_severity', lambda s: float(s.dropna().max()) if s.dropna().shape[0]>0 else np.nan),
        avg_max_severity = ('max_severity', lambda s: float(s.dropna().mean()) if s.dropna().shape[0]>0 else np.nan),
        earliest_fault_start_ts = ('fault_start_ts', 'min')
    ).reset_index()

    # For each cmdb_id get top 3 worst kpi_name by max_severity
    # Build a helper mapping of cmdb_id -> list of top kpis
    worst_kpis = {}
    for cmdb, g in subset.groupby('cmdb_id'):
        g2 = g.copy()
        g2['max_severity'] = pd.to_numeric(g2['max_severity'], errors='coerce')
        # Sort descending by max_severity (NaNs go last)
        g2 = g2.sort_values('max_severity', ascending=False)
        topk = g2.head(3)[['kpi_name', 'max_severity']].to_dict(orient='records')
        # Round severities to 6 decimals and ensure serializable
        formatted = []
        for rec in topk:
            kpi = str(rec['kpi_name'])
            sev = rec['max_severity']
            sev = float(round(sev,6)) if pd.notna(sev) else None
            formatted.append({'kpi_name': kpi, 'max_severity': sev})
        worst_kpis[cmdb] = formatted

    # Merge worst_kpis into agg and prepare rows
    agg['fault_start_ts'] = int(ts)
    agg['max_max_severity'] = agg['max_max_severity'].apply(lambda x: round(float(x),6) if pd.notna(x) else None)
    agg['avg_max_severity'] = agg['avg_max_severity'].apply(lambda x: round(float(x),6) if pd.notna(x) else None)
    agg['earliest_fault_start_ts'] = agg['earliest_fault_start_ts'].astype(int)

    agg = agg.sort_values(['faults_in_cluster', 'max_max_severity'], ascending=[False, False]).head(10).reset_index(drop=True)

    for _, r in agg.iterrows():
        cmdb = r['cmdb_id']
        candidates_rows.append({
            'fault_start_ts': int(ts),
            'cmdb_id': str(cmdb),
            'faults_in_cluster': int(r['faults_in_cluster']),
            'distinct_kpi_count': int(r['distinct_kpi_count']),
            'max_max_severity': r['max_max_severity'],
            'avg_max_severity': r['avg_max_severity'],
            'earliest_fault_start_ts': int(r['earliest_fault_start_ts']),
            'sample_worst_kpis': worst_kpis.get(cmdb, [])  # up to 3 entries
        })

# Build DataFrame of candidates
top_candidates_per_cluster = pd.DataFrame(candidates_rows, columns=[
    'fault_start_ts', 'cmdb_id', 'faults_in_cluster', 'distinct_kpi_count',
    'max_max_severity', 'avg_max_severity', 'earliest_fault_start_ts', 'sample_worst_kpis'
])

# Final sanity: ensure types and rounding
for col in ['max_max_severity', 'avg_max_severity']:
    if col in top_candidates_per_cluster.columns:
        top_candidates_per_cluster[col] = top_candidates_per_cluster[col].apply(lambda x: round(float(x),6) if x is not None else None)

# Results stored in kernel variables:
top_clusters_by_count, top_candidates_per_cluster

```
Out[6]:
```
Summary of results (plain English):

1) Top fault-start clusters by count
- The largest cluster starts at 1647748800 (2022-03-20 12:00:00 Asia/Shanghai) with 3,711 faults across 59 distinct cmdb_id values — a massive, system-wide surge at the window start.
- The next biggest clusters are:
  - 1647749520: 66 faults (20 distinct cmdb_ids)
  - 1647749340: 53 faults (12 distinct cmdb_ids)
  - 1647748980: 53 faults (28 distinct cmdb_ids)
  - 1647749700: 53 faults (20 distinct cmdb_ids)
  - 1647749220: 47 faults (16 distinct cmdb_ids)
  - 1647748920: 46 faults (24 distinct cmdb_ids)
  - 1647749880: 39 faults (17 distinct cmdb_ids)
  - 1647749820: 38 faults (20 distinct cmdb_ids)
  - 1647749040: 33 faults (15 distinct cmdb_ids)

(These top-10 cluster rows are stored in kernel variable top_clusters_by_count.)

2) Top candidate components inside the two largest clusters
- For the largest cluster at 1647748800 (3711 faults), the top cmdb_id candidates (top 10 by faults in cluster) include:
  - adservice: 304 faults (304 distinct KPIs), max_max_severity ≈ 1.016667, avg_max_severity ≈ 0.989802. Sample worst KPIs: various JVM/runtime KPIs.
  - adservice2: 290 faults, max_max_severity ≈ 1.016667, avg ≈ 0.988907.
  - frontend-0: 129 faults, max_max_severity ≈ 1.036956, avg ≈ 1.002704. Sample worst KPI: mesh source to cartservice.
  - frontend2-0: 126 faults, max_max_severity = 1.000000, avg = 1.000000. Sample worst KPI: container filesystem metrics.
  - frontend-1, frontend-2: each ~123 faults, max_max_severity ≈ 1.0.
  - productcatalogservice-2 / productcatalogservice-0 / productcatalogservice-1: 93, 88, 80 faults respectively, max_max_severity in the ~1.006–1.018 range.
  - checkoutservice-1: 81 faults, max_max_severity ≈ 1.306773 (notably higher).
  - (Each entry’s earliest_fault_start_ts equals the cluster ts 1647748800. Short samples of the worst KPIs per cmdb_id are included in the stored data.)

- For the second cluster at 1647749520 (66 faults), top cmdb_id candidates include:
  - checkoutservice-0: 22 faults, max_max_severity = 1.0, avg = 1.0.
  - recommendationservice-2: 7 faults, max_max_severity ≈ 1.475960, avg ≈ 1.112441.
  - recommendationservice-0: 6 faults, max_max_severity ≈ 1.737254, avg ≈ 1.141302 (highest single cmdb max in this cluster).
  - recommendationservice-1: 4 faults, max_max_severity ≈ 1.449210.
  - node-6: 4 faults, max_max_severity ≈ 1.169960 (system metrics such as load visible).
  - productcatalogservice-0, recommendationservice2-0, productcatalogservice-1, checkoutservice-1, adservice2-0 follow with fewer faults but elevated per-kpi severities in some cases.
  - Sample worst KPIs for these cmdb_ids include container memory metrics, mesh destination metrics, and system.load — these are included in the stored sample lists.

(These candidate summaries are stored in kernel variable top_candidates_per_cluster. Each candidate row includes: fault_start_ts, cmdb_id, faults_in_cluster, distinct_kpi_count, max_max_severity, avg_max_severity, earliest_fault_start_ts, and up to 3 worst kpi_name / max_severity samples.)

Key takeaways
- The incident is highly concentrated at 12:00:00 (1647748800) with thousands of faults across many services — indicating a broad system-wide disturbance at the window start.
- Within that large cluster, frontend and adservice/productcatalogservice families show the largest numbers of faults; checkoutservice-1 shows relatively higher per-kpi severity in that cluster.
- A smaller but notable cluster at 1647749520 highlights recommendationservice instances and node-6 with elevated severities (some max_max_severity values >1.4–1.7), suggesting memory or node-level issues there.
- Full compact outputs are stored in kernel variables top_clusters_by_count and top_candidates_per_cluster for further investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(   fault_start_ts  fault_count  distinct_cmdb_count                                   example_cmdb_ids
0      1647748800         3711                   59  [adservice, adservice-0, adservice-1, adservic...
1      1647749520           66                   20  [adservice, adservice-1, adservice2-0, checkou...
2      1647749340           53                   12  [adservice-2, cartservice2-0, checkoutservice-...
3      1647748980           53                   28  [adservice-2, cartservice-0, checkoutservice-0...
4      1647749700           53                   20  [adservice, cartservice-1, cartservice2-0, che...
5      1647749220           47                   16  [adservice, adservice-0, cartservice-0, checko...
6      1647748920           46                   24  [adservice, adservice-0, cartservice, cartserv...
7      1647749880           39                   17  [adservice, checkoutservice-0, emailservice-0,...
8      1647749820           38                   20  [adservice, adservice-0, adservice2, adservice...
9      1647749040           33                   15  [adservice2-0, cartservice-2, checkoutservice-...,     fault_start_ts                   cmdb_id  faults_in_cluster  distinct_kpi_count  max_max_severity  avg_max_severity  earliest_fault_start_ts                                  sample_worst_kpis
0       1647748800                 adservice                304                 304          1.016667          0.989802               1647748800  [{'kpi_name': 'runtime.jvm_threads_state.RUNNA...
1       1647748800                adservice2                290                 290          1.016667          0.988907               1647748800  [{'kpi_name': 'runtime.jvm_threads_state.RUNNA...
2       1647748800                frontend-0                129                 129          1.036956          1.002704               1647748800  [{'kpi_name': 'mesh.source.frontend.cartservic...
3       1647748800               frontend2-0                126                 126          1.000000          1.000000               1647748800  [{'kpi_name': 'container.node-6.container_fs_l...
4       1647748800                frontend-1                123                 123          1.000000          0.805454               1647748800  [{'kpi_name': 'container.node-5.container_fs_l...
5       1647748800                frontend-2                123                 123          1.000000          0.888889               1647748800  [{'kpi_name': 'container.node-5.container_file...
6       1647748800   productcatalogservice-2                 93                  93          1.018348          0.974331               1647748800  [{'kpi_name': 'mesh.destination.frontend.produ...
7       1647748800   productcatalogservice-0                 88                  88          1.006762          0.930559               1647748800  [{'kpi_name': 'mesh.destination.frontend.produ...
8       1647748800         checkoutservice-1                 81                  81          1.306773          1.030111               1647748800  [{'kpi_name': 'mesh.source.checkoutservice.pay...
9       1647748800   productcatalogservice-1                 80                  80          1.125000          0.923144               1647748800  [{'kpi_name': 'container.node-6.container_sock...
10      1647749520         checkoutservice-0                 22                  22          1.000000          1.000000               1647749520  [{'kpi_name': 'mesh.destination.frontend.check...
11      1647749520   recommendationservice-2                  7                   7          1.475960          1.112441               1647749520  [{'kpi_name': 'container.node-6.container_memo...
12      1647749520   recommendationservice-0                  6                   6          1.737254          1.141302               1647749520  [{'kpi_name': 'container.node-6.container_memo...
13      1647749520   recommendationservice-1                  4                   4          1.449210          1.138457               1647749520  [{'kpi_name': 'container.node-6.container_memo...
14      1647749520                    node-6                  4                   4          1.169960          0.292490               1647749520  [{'kpi_name': 'system.load.15', 'max_severity'...
15      1647749520   productcatalogservice-0                  3                   3          2.465257          1.902146               1647749520  [{'kpi_name': 'mesh.destination.recommendation...
16      1647749520  recommendationservice2-0                  3                   3          1.954237          1.420274               1647749520  [{'kpi_name': 'container.node-6.container_memo...
17      1647749520   productcatalogservice-1                  2                   2          1.927027          1.677342               1647749520  [{'kpi_name': 'mesh.destination.recommendation...
18      1647749520         checkoutservice-1                  2                   2          0.222222          0.114073               1647749520  [{'kpi_name': 'container.node-6.container_cpu_...
19      1647749520              adservice2-0                  2                   2          0.166667          0.166667               1647749520  [{'kpi_name': 'container.node-6.container_memo...)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse trace_incident_df from kernel
cluster_ts_list = [1647748800, 1647749520]

def analyze_trace_cluster(cluster_ts, window_seconds=600, top_n_groups=20, top_n_cmdb=10):
    start = int(cluster_ts)
    end = int(cluster_ts + window_seconds)
    win = trace_incident_df[(trace_incident_df['timestamp'] >= start) & (trace_incident_df['timestamp'] <= end)].copy()
    if win.empty:
        cols_g = ['cmdb_id','trace_name','total_points','anomaly_count_high','anomaly_count_low','earliest_anomaly_ts','max_severity','mean_value']
        cols_c = ['cmdb_id','total_trace_groups_in_window','total_anomalous_points','max_max_severity','earliest_anomaly_ts']
        return pd.DataFrame(columns=cols_g), pd.DataFrame(columns=cols_c)
    win['value'] = pd.to_numeric(win['value'], errors='coerce')
    win['p95'] = pd.to_numeric(win['p95'], errors='coerce')
    win['p5'] = pd.to_numeric(win['p5'], errors='coerce')
    win['is_high'] = (~win['p95'].isna()) & (win['value'] >= win['p95'])
    win['is_low'] = (~win['p5'].isna()) & (win['value'] <= win['p5'])
    win['is_anom'] = win['is_high'] | win['is_low']
    win['severity'] = np.where((~win['p95'].isna()) & (win['p95'] > 0), win['value'] / win['p95'], np.nan)
    win['anom_ts'] = np.where(win['is_anom'], win['timestamp'], np.nan)

    grp = win.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        total_points = ('timestamp','size'),
        anomaly_count_high = ('is_high','sum'),
        anomaly_count_low = ('is_low','sum'),
        earliest_anomaly_ts = ('anom_ts','min'),
        max_severity = ('severity','max'),
        mean_value = ('value','mean')
    )

    grp['earliest_anomaly_ts'] = grp['earliest_anomaly_ts'].apply(lambda x: int(x) if pd.notna(x) else None)
    grp['max_severity'] = grp['max_severity'].round(6)
    grp['mean_value'] = grp['mean_value'].round(6)
    grp['total_points'] = grp['total_points'].astype(int)
    grp['anomaly_count_high'] = grp['anomaly_count_high'].astype(int)
    grp['anomaly_count_low'] = grp['anomaly_count_low'].astype(int)
    grp['anomaly_sum'] = grp['anomaly_count_high'] + grp['anomaly_count_low']

    grp_sorted = grp.sort_values(['anomaly_sum','max_severity'], ascending=[False, False]).reset_index(drop=True)
    top_groups = grp_sorted.head(top_n_groups)[[
        'cmdb_id','trace_name','total_points','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_ts','max_severity','mean_value'
    ]].copy()

    # Candidates aggregated by cmdb_id
    cand_rows = []
    for cmdb, sub in grp.groupby('cmdb_id'):
        total_groups = int(sub.shape[0])
        total_anoms = int((sub['anomaly_count_high'] + sub['anomaly_count_low']).sum())
        max_max_sev = sub['max_severity'].dropna().max() if sub['max_severity'].dropna().shape[0] > 0 else np.nan
        earliest_anom = sub['earliest_anomaly_ts'].dropna()
        earliest_anom = int(earliest_anom.min()) if earliest_anom.shape[0] > 0 else None
        cand_rows.append({
            'cmdb_id': cmdb,
            'total_trace_groups_in_window': total_groups,
            'total_anomalous_points': total_anoms,
            'max_max_severity': round(float(max_max_sev),6) if pd.notna(max_max_sev) else None,
            'earliest_anomaly_ts': earliest_anom
        })
    candidates_df = pd.DataFrame(cand_rows)
    if not candidates_df.empty:
        candidates_df = candidates_df.sort_values(['total_anomalous_points','max_max_severity'], ascending=[False, False]).head(top_n_cmdb).reset_index(drop=True)

    return top_groups, candidates_df

# Run analysis for both clusters and store results in kernel variables
trace_cluster_1647748800, trace_candidates_1647748800 = analyze_trace_cluster(1647748800)
trace_cluster_1647749520, trace_candidates_1647749520 = analyze_trace_cluster(1647749520)

trace_candidates_per_cluster = {
    1647748800: trace_candidates_1647748800,
    1647749520: trace_candidates_1647749520
}

# Return compact results
trace_cluster_1647748800, trace_cluster_1647749520, trace_candidates_per_cluster

```
Out[7]:
```
Summary of trace analysis (10-minute windows) for the two clusters:

Window definition: for each cluster_ts we examined trace_incident_df in [cluster_ts, cluster_ts + 600] (inclusive). Results stored in kernel variables:
- trace_cluster_1647748800 (top 20 (cmdb_id,trace_name) groups for ts=1647748800)
- trace_cluster_1647749520 (top 20 groups for ts=1647749520)
- trace_candidates_per_cluster (top-cmdb summaries for each cluster)

Key findings — cluster 1647748800:
- Many trace groups show all points anomalous in the 10-min window (e.g., numerous trace.from_frontend-*.error_rate groups across adservice, cartservice, etc. with total_points=11 and anomaly_count_high=11 and anomaly_count_low=11). For many error_rate series p95==0 so max_severity is NaN.
- A few groups have numeric max_severity=1.0 for duration_p95 series (e.g., cartservice2-0 trace.from_frontend2-0.duration_p95 and frontend2-0 trace.to_cartservice2-0.duration_p95).
- Candidate cmdb_ids (top 10 by total anomalous points) — highest contributors:
  - checkoutservice-0: total_trace_groups_in_window=92, total_anomalous_points=806, max_max_severity≈18.248175 (earliest_anomaly_ts=1647748800)
  - checkoutservice-2: 92 groups, 755 anomalous points, max_max_severity≈4.205556
  - checkoutservice-1: 92 groups, 734 anomalous points, max_max_severity≈1.880381
  - frontend-0 / frontend-1 / frontend-2: each ~92 groups, hundreds of anomalous points, max_max_severity between ~1.42–1.58
  - cartservice-1 / cartservice-0 / cartservice-2 and productcatalogservice-1 also appear in the top-10 contributors.
- Interpretation: checkoutservice and frontends are dominant sources of trace anomalies in this window; some checkoutservice-related trace KPIs show very large severity ratios (>4×, up to ~18× p95).

Key findings — cluster 1647749520:
- Similar pattern: many trace.from_frontend-*.error_rate groups are fully anomalous (total_points=11, anomaly_count_high=11, anomaly_count_low=11) and report NaN severity where p95==0.
- Duration_p95 groups again appear with max_severity=1.0 in some places.
- Candidate cmdb_ids (top 10 by anomalous points) — highest contributors:
  - checkoutservice-0: 92 groups, 816 anomalous points, max_max_severity≈12.524904 (earliest_anomaly_ts=1647749520)
  - checkoutservice-1: 92 groups, 771 anomalous points, max_max_severity≈31.457286
  - checkoutservice-2: 92 groups, 725 anomalous points, max_max_severity≈67.139785 (very high)
  - frontends and cartservices again appear with high totals; frontend-2 shows a notable max_max_severity (~6.738916).
- Interpretation: second cluster shows even larger per-KPI severities on some checkoutservice instances (checkoutservice-2 up to ~67× p95), indicating strong trace-level degradations around 1647749520.

Overall conclusions:
- Both clusters show heavy and widespread trace anomalies concentrated in the 10-minute windows after cluster start.
- Checkoutservice family and frontend pods are major contributors in both windows; checkoutservice instances show the highest per-KPI severities (notably for the second cluster).
- Many error_rate series are anomalous but have undefined severity where p95==0 — use counts/timing to evaluate these.
- All returned compact tables are saved in kernel variables for follow-up: trace_cluster_1647748800, trace_cluster_1647749520, and trace_candidates_per_cluster.

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                                  trace_name  total_points  anomaly_count_high  anomaly_count_low  earliest_anomaly_ts  max_severity  mean_value
0   cartservice2-0         trace.from_frontend2-0.duration_p95            11                  11                 11         1.647749e+09           1.0       0.001
1      frontend2-0        trace.to_cartservice2-0.duration_p95            11                  11                 11         1.647749e+09           1.0       0.001
2      adservice-0            trace.from_frontend-0.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
3      adservice-0            trace.from_frontend-1.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
4      adservice-0            trace.from_frontend-2.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
5      adservice-1            trace.from_frontend-0.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
6      adservice-1            trace.from_frontend-1.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
7      adservice-1            trace.from_frontend-2.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
8      adservice-2            trace.from_frontend-0.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
9      adservice-2            trace.from_frontend-1.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
10     adservice-2            trace.from_frontend-2.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
11    adservice2-0           trace.from_frontend2-0.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
12   cartservice-0  trace.from_checkoutservice-0.duration_mean            11                  11                 11         1.647749e+09           NaN       0.000
13   cartservice-0   trace.from_checkoutservice-0.duration_p95            11                  11                 11         1.647749e+09           NaN       0.000
14   cartservice-0     trace.from_checkoutservice-0.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
15   cartservice-0     trace.from_checkoutservice-1.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
16   cartservice-0     trace.from_checkoutservice-2.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
17   cartservice-0            trace.from_frontend-0.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
18   cartservice-0            trace.from_frontend-1.error_rate            11                  11                 11         1.647749e+09           NaN       0.000
19   cartservice-0            trace.from_frontend-2.error_rate            11                  11                 11         1.647749e+09           NaN       0.000,            cmdb_id                               trace_name  total_points  anomaly_count_high  anomaly_count_low  earliest_anomaly_ts  max_severity  mean_value
0   cartservice2-0      trace.from_frontend2-0.duration_p95            11                  11                 11         1.647750e+09           1.0       0.001
1      frontend2-0     trace.to_cartservice2-0.duration_p95            11                  11                 11         1.647750e+09           1.0       0.001
2      adservice-0         trace.from_frontend-0.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
3      adservice-0         trace.from_frontend-1.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
4      adservice-0         trace.from_frontend-2.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
5      adservice-1         trace.from_frontend-0.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
6      adservice-1         trace.from_frontend-1.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
7      adservice-1         trace.from_frontend-2.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
8      adservice-2         trace.from_frontend-0.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
9      adservice-2         trace.from_frontend-2.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
10    adservice2-0        trace.from_frontend2-0.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
11   cartservice-0  trace.from_checkoutservice-0.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
12   cartservice-0  trace.from_checkoutservice-1.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
13   cartservice-0  trace.from_checkoutservice-2.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
14   cartservice-0         trace.from_frontend-0.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
15   cartservice-0         trace.from_frontend-1.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
16   cartservice-0         trace.from_frontend-2.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
17   cartservice-0                  trace.self.duration_p95            11                  11                 11         1.647750e+09           NaN       0.000
18   cartservice-0                    trace.self.error_rate            11                  11                 11         1.647750e+09           NaN       0.000
19   cartservice-1  trace.from_checkoutservice-0.error_rate            11                  11                 11         1.647750e+09           NaN       0.000, {1647748800:                    cmdb_id  total_trace_groups_in_window  total_anomalous_points  max_max_severity  earliest_anomaly_ts
0        checkoutservice-0                            92                     806         18.248175           1647748800
1        checkoutservice-2                            92                     755          4.205556           1647748800
2        checkoutservice-1                            92                     734          1.880381           1647748800
3               frontend-0                            92                     674          1.576336           1647748800
4               frontend-1                            92                     643          1.483871           1647748800
5               frontend-2                            92                     636          1.422222           1647748800
6            cartservice-1                            28                     353          1.200000           1647748800
7            cartservice-0                            28                     347          1.331288           1647748800
8            cartservice-2                            28                     311         18.248175           1647748800
9  productcatalogservice-1                            40                     285          1.880381           1647748800, 1647749520:                    cmdb_id  total_trace_groups_in_window  total_anomalous_points  max_max_severity  earliest_anomaly_ts
0        checkoutservice-0                            92                     816         12.524904           1647749520
1        checkoutservice-1                            92                     771         31.457286           1647749520
2        checkoutservice-2                            92                     725         67.139785           1647749520
3               frontend-0                            92                     679          1.482204           1647749520
4               frontend-2                            92                     651          6.738916           1647749520
5               frontend-1                            92                     634          2.430851           1647749520
6            cartservice-1                            28                     361          6.738916           1647749520
7            cartservice-0                            28                     339          5.464481           1647749520
8            cartservice-2                            28                     306         27.272727           1647749520
9  productcatalogservice-2                            40                     295          2.733333           1647749520})```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse kernel variables: log_incident_df, log_thresholds, error_logs_df

def analyze_log_cluster(cluster_ts, window_seconds=600, top_n_groups=20, top_n_cmdb=20):
    start = int(cluster_ts)
    end = int(cluster_ts + window_seconds)
    win = log_incident_df[(log_incident_df['timestamp'] >= start) & (log_incident_df['timestamp'] <= end)].copy()
    if win.empty:
        empty_groups = pd.DataFrame(columns=['cmdb_id','log_name','total_points','anomaly_count_high','anomaly_count_low','earliest_anomaly_ts','mean_value','max_value'])
        empty_cmdbs = pd.DataFrame(columns=['cmdb_id','total_log_groups','total_anomalous_points','earliest_anomaly_ts','any_log_error_count_detected'])
        return empty_groups, empty_cmdbs, pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    # Ensure numeric
    win['value'] = pd.to_numeric(win['value'], errors='coerce')
    # Ensure thresholds present by merging global thresholds if needed
    if not {'p95','p5'}.issubset(win.columns):
        win = win.merge(log_thresholds[['cmdb_id','log_name','p95','p5']], on=['cmdb_id','log_name'], how='left')
    else:
        # If present but may have NaNs, fill from global where missing
        missing_thresh = win['p95'].isna() | win['p5'].isna()
        if missing_thresh.any():
            win = win.drop(columns=['p95','p5']).merge(log_thresholds[['cmdb_id','log_name','p95','p5']], on=['cmdb_id','log_name'], how='left')
    # Compute anomaly masks
    win['is_high'] = (~win['p95'].isna()) & (win['value'] >= win['p95'])
    win['is_low'] = (~win['p5'].isna()) & (win['value'] <= win['p5'])
    win['is_anom'] = win['is_high'] | win['is_low']
    # Group basic aggregates
    grp_counts = win.groupby(['cmdb_id','log_name'], as_index=False).agg(
        total_points = ('timestamp','size'),
        anomaly_count_high = ('is_high','sum'),
        anomaly_count_low = ('is_low','sum')
    )
    # earliest anomaly timestamp per group
    earliest = win[win['is_anom']].groupby(['cmdb_id','log_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_ts'})
    grp = grp_counts.merge(earliest, on=['cmdb_id','log_name'], how='left')
    grp['earliest_anomaly_ts'] = grp['earliest_anomaly_ts'].apply(lambda x: int(x) if pd.notna(x) else None)
    # mean and max value
    stats = win.groupby(['cmdb_id','log_name'], as_index=False).agg(
        mean_value = ('value','mean'),
        max_value = ('value','max')
    )
    grp = grp.merge(stats, on=['cmdb_id','log_name'], how='left')
    grp['mean_value'] = grp['mean_value'].round(6)
    grp['max_value'] = grp['max_value'].round(6)
    grp['total_points'] = grp['total_points'].astype(int)
    grp['anomaly_count_high'] = grp['anomaly_count_high'].astype(int)
    grp['anomaly_count_low'] = grp['anomaly_count_low'].astype(int)
    # Top groups sorted by anomaly_count_high desc, then anomaly_count_low desc
    top_groups = grp.sort_values(['anomaly_count_high','anomaly_count_low'], ascending=[False, False]).head(top_n_groups)
    top_groups = top_groups[['cmdb_id','log_name','total_points','anomaly_count_high','anomaly_count_low','earliest_anomaly_ts','mean_value','max_value']].reset_index(drop=True)
    # Per-cmdb aggregates
    grp['anom_sum'] = grp['anomaly_count_high'] + grp['anomaly_count_low']
    cmdb_agg = grp.groupby('cmdb_id', as_index=False).agg(
        total_log_groups = ('log_name','nunique'),
        total_anomalous_points = ('anom_sum','sum'),
        earliest_anomaly_ts = ('earliest_anomaly_ts','min')
    )
    cmdb_agg['earliest_anomaly_ts'] = cmdb_agg['earliest_anomaly_ts'].apply(lambda x: int(x) if pd.notna(x) else None)
    # any_log_error_count_detected for log.error_count groups
    error_counts = grp.loc[grp['log_name']=='log.error_count', ['cmdb_id','max_value']].copy()
    if not error_counts.empty:
        error_counts['max_value'] = pd.to_numeric(error_counts['max_value'], errors='coerce').fillna(0)
        error_flag = error_counts.groupby('cmdb_id', as_index=False).agg(any_detected = ('max_value', lambda s: bool((s>0).any())))
    else:
        error_flag = pd.DataFrame(columns=['cmdb_id','any_detected'])
    cmdb_agg = cmdb_agg.merge(error_flag, on='cmdb_id', how='left')
    cmdb_agg['any_log_error_count_detected'] = cmdb_agg['any_detected'].fillna(False).astype(bool)
    cmdb_agg = cmdb_agg.drop(columns=['any_detected'])
    cmdb_agg = cmdb_agg.sort_values('total_anomalous_points', ascending=False).head(top_n_cmdb).reset_index(drop=True)
    # Error logs in window
    if 'error_logs_df' in globals():
        err_in_window = error_logs_df[(error_logs_df['timestamp'] >= start) & (error_logs_df['timestamp'] <= end)].copy()
        if not err_in_window.empty:
            err_in_window = err_in_window[['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)
        else:
            err_in_window = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    else:
        err_in_window = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    return top_groups, cmdb_agg, err_in_window

# Run for both clusters and store kernel variables
top_groups_1647748800, cmdb_agg_1647748800, errlogs_1647748800 = analyze_log_cluster(1647748800)
log_cluster_1647748800 = {
    'top_groups': top_groups_1647748800,
    'cmdb_aggregates': cmdb_agg_1647748800,
    'error_logs': errlogs_1647748800
}

top_groups_1647749520, cmdb_agg_1647749520, errlogs_1647749520 = analyze_log_cluster(1647749520)
log_cluster_1647749520 = {
    'top_groups': top_groups_1647749520,
    'cmdb_aggregates': cmdb_agg_1647749520,
    'error_logs': errlogs_1647749520
}

# Expose results
log_cluster_1647748800, log_cluster_1647749520

```
Out[8]:
```
summary = (
    "Log analysis for two clusters (10‑minute windows starting at 1647748800 and 1647749520):\n\n"
    "1) Top log groups (both windows):\n"
    "- The top groups are overwhelmingly log.error_count across many services (e.g., adservice-0/1/2, cartservice-*, checkoutservice-*, frontend-*, etc.).\n"
    "- For these groups total_points=11 and anomaly_count_high=11 (and anomaly_count_low=11 in some groupings), but mean_value and max_value are 0.0.\n"
    "- This indicates thresholds (p95) are 0, so values of 0 are considered anomalous by the >=p95 rule, producing many flagged groups with zero observed errors.\n\n"
    "2) Per-cmdb aggregates (top contributors):\n"
    "- In the 1647748800 window, many services (emailservice, paymentservice, cartservice, recommendationservice, productcatalogservice, frontend, shippingservice, adservice, checkoutservice, etc.) show dozens of anomalous points each; any_log_error_count_detected is False for these (no non-zero log.error_count seen).\n"
    "- In the 1647749520 window, a similar pattern appears with slightly different ordering of top cmdbs; again no actual log.error_count > 0 detected in the returned aggregates.\n\n"
    "3) Error logs: none found in either window (empty result).\n\n"
    "Conclusion / next steps:\n"
    "- There is a large number of log.error_count anomalies driven by zero-valued series where p95==0; this is likely an artifact of thresholding rather than true logged errors. Investigate threshold calculation for log.error_count (p95==0) and confirm whether zero-values should be treated as anomalies. Also correlate with other signals (metrics/traces) and with actual non-zero error occurrences to prioritize follow-up."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

({'top_groups':               cmdb_id         log_name  total_points  anomaly_count_high  anomaly_count_low  earliest_anomaly_ts  mean_value  max_value
0         adservice-0  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
1         adservice-1  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
2         adservice-2  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
3       cartservice-0  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
4       cartservice-1  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
5       cartservice-2  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
6   checkoutservice-0  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
7   checkoutservice-1  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
8   checkoutservice-2  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
9   currencyservice-0  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
10  currencyservice-1  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
11  currencyservice-2  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
12     emailservice-0  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
13     emailservice-1  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
14     emailservice-2  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
15         frontend-0  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
16         frontend-1  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
17         frontend-2  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
18   paymentservice-0  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0
19   paymentservice-1  log.error_count            11                  11                 11         1.647749e+09         0.0        0.0, 'cmdb_aggregates':                     cmdb_id  total_log_groups  total_anomalous_points  earliest_anomaly_ts  any_log_error_count_detected
0            emailservice-1                 2                      25           1647748800                         False
1            emailservice-2                 2                      25           1647748800                         False
2          paymentservice-2                 2                      25           1647748800                         False
3          paymentservice-1                 2                      25           1647748800                         False
4            emailservice-0                 2                      24           1647748800                         False
5             cartservice-2                 2                      24           1647748800                         False
6             cartservice-1                 2                      24           1647748800                         False
7   recommendationservice-0                 2                      24           1647748800                         False
8   recommendationservice-1                 2                      24           1647748800                         False
9   recommendationservice-2                 2                      24           1647748800                         False
10  productcatalogservice-1                 2                      24           1647748800                         False
11  productcatalogservice-0                 2                      24           1647748800                         False
12               frontend-1                 2                      24           1647748800                         False
13        shippingservice-2                 2                      24           1647748800                         False
14        shippingservice-1                 2                      24           1647748800                         False
15              adservice-0                 2                      23           1647748800                         False
16              adservice-2                 2                      23           1647748800                         False
17               frontend-0                 2                      23           1647748800                         False
18  productcatalogservice-2                 2                      23           1647748800                         False
19        checkoutservice-1                 2                      23           1647748800                         False, 'error_logs': Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: []}, {'top_groups':               cmdb_id         log_name  total_points  anomaly_count_high  anomaly_count_low  earliest_anomaly_ts  mean_value  max_value
0         adservice-0  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
1         adservice-1  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
2         adservice-2  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
3       cartservice-0  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
4       cartservice-1  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
5       cartservice-2  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
6   checkoutservice-0  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
7   checkoutservice-1  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
8   checkoutservice-2  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
9   currencyservice-0  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
10  currencyservice-1  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
11  currencyservice-2  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
12     emailservice-0  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
13     emailservice-1  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
14     emailservice-2  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
15         frontend-0  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
16         frontend-1  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
17         frontend-2  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
18   paymentservice-0  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0
19   paymentservice-1  log.error_count            11                  11                 11         1.647750e+09         0.0        0.0, 'cmdb_aggregates':                     cmdb_id  total_log_groups  total_anomalous_points  earliest_anomaly_ts  any_log_error_count_detected
0          paymentservice-1                 2                      27           1647749520                         False
1            emailservice-0                 2                      25           1647749520                         False
2               adservice-2                 2                      25           1647749520                         False
3          paymentservice-0                 2                      25           1647749520                         False
4   productcatalogservice-2                 2                      25           1647749520                         False
5         checkoutservice-1                 2                      25           1647749520                         False
6         checkoutservice-0                 2                      24           1647749520                         False
7         currencyservice-1                 2                      24           1647749520                         False
8         currencyservice-2                 2                      24           1647749520                         False
9         checkoutservice-2                 2                      24           1647749520                         False
10               frontend-2                 2                      24           1647749520                         False
11  productcatalogservice-1                 2                      24           1647749520                         False
12        currencyservice-0                 2                      24           1647749520                         False
13              adservice-0                 2                      24           1647749520                         False
14              adservice-1                 2                      24           1647749520                         False
15           emailservice-2                 2                      23           1647749520                         False
16           emailservice-1                 2                      23           1647749520                         False
17            cartservice-1                 2                      23           1647749520                         False
18            cartservice-2                 2                      23           1647749520                         False
19            cartservice-0                 2                      23           1647749520                         False, 'error_logs': Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: []})```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse kernel variables: trace_incident_df, metric_faults (assumed present)

cluster_ts_list = [1647748800, 1647749520]

def analyze_cluster_downstream_and_metrics(cluster_ts, trace_df, metric_faults_df):
    start = int(cluster_ts)
    end = int(cluster_ts + 600)
    # Filter trace rows in window
    win = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
    # Ensure numeric and threshold cols
    win['value'] = pd.to_numeric(win['value'], errors='coerce')
    win['p95'] = pd.to_numeric(win.get('p95'), errors='coerce')
    win['p5'] = pd.to_numeric(win.get('p5'), errors='coerce')
    # Mark anomalous: value >= p95 OR value <= p5 (handle NaNs -> False)
    win['is_high'] = (~win['p95'].isna()) & (win['value'] >= win['p95'])
    win['is_low'] = (~win['p5'].isna()) & (win['value'] <= win['p5'])
    win['is_anom'] = win['is_high'] | win['is_low']
    # Only anomalous rows needed for counts
    anoms = win[win['is_anom']].copy()
    # incoming/outgoing: consider distinct trace_name groups containing substrings
    anoms['is_incoming'] = anoms['trace_name'].astype(str).str.contains('trace.from_', na=False)
    anoms['is_outgoing'] = anoms['trace_name'].astype(str).str.contains('trace.to_', na=False)
    # Per-cmdb aggregates
    # incoming_anomalous_groups: distinct trace_name with 'trace.from_' that are anomalous
    incoming = anoms[anoms['is_incoming']].groupby('cmdb_id')['trace_name'].nunique().rename('incoming_anomalous_groups')
    outgoing = anoms[anoms['is_outgoing']].groupby('cmdb_id')['trace_name'].nunique().rename('outgoing_anomalous_groups')
    total_points = anoms.groupby('cmdb_id').size().rename('total_anomalous_points')
    earliest = anoms.groupby('cmdb_id')['timestamp'].min().rename('earliest_anomaly_ts')
    # Combine into DataFrame
    downstream = pd.concat([incoming, outgoing, total_points, earliest], axis=1).fillna(0)
    downstream['incoming_anomalous_groups'] = downstream['incoming_anomalous_groups'].astype(int)
    downstream['outgoing_anomalous_groups'] = downstream['outgoing_anomalous_groups'].astype(int)
    downstream['total_anomalous_points'] = downstream['total_anomalous_points'].astype(int)
    downstream['earliest_anomaly_ts'] = downstream['earliest_anomaly_ts'].apply(lambda x: int(x) if x and not pd.isna(x) else None)
    downstream = downstream.reset_index()
    # Sort as requested: incoming desc, tie-breaker total_anomalous_points desc
    downstream_sorted = downstream.sort_values(['incoming_anomalous_groups','total_anomalous_points'], ascending=[False, False]).head(20).reset_index(drop=True)
    # Step 2: metric evidence for top 10 cmdb_id candidates
    top10_cmdbs = downstream_sorted['cmdb_id'].head(10).tolist()
    candidate_agg_rows = []
    candidate_faults_rows = []
    # prepare metric_faults_df numeric rounding
    mf = metric_faults_df.copy()
    # ensure numeric severities
    mf['max_severity'] = pd.to_numeric(mf.get('max_severity'), errors='coerce')
    mf['min_severity'] = pd.to_numeric(mf.get('min_severity'), errors='coerce')
    for cmdb in top10_cmdbs:
        sub = mf[mf['cmdb_id'] == cmdb].copy()
        if sub.empty:
            candidate_agg_rows.append({
                'cmdb_id': cmdb,
                'highest_max_severity_across_faults': None,
                'earliest_fault_start_ts': None,
                'number_of_faults_in_metric_faults': 0
            })
            continue
        # aggregate row
        highest = sub['max_severity'].dropna()
        highest_val = float(highest.max()) if highest.shape[0] > 0 else None
        earliest_fault = int(sub['fault_start_ts'].min())
        num_faults = int(sub.shape[0])
        candidate_agg_rows.append({
            'cmdb_id': cmdb,
            'highest_max_severity_across_faults': round(highest_val,6) if highest_val is not None else None,
            'earliest_fault_start_ts': earliest_fault,
            'number_of_faults_in_metric_faults': num_faults
        })
        # top 5 faults sorted by max_severity desc (NaNs last)
        sub_sorted = sub.copy()
        # For sorting, treat NaN as -inf so they go last
        sub_sorted['_sort_sev'] = sub_sorted['max_severity'].fillna(-np.inf)
        sub_sorted = sub_sorted.sort_values(['_sort_sev','points'], ascending=[False, False]).head(5)
        # select requested fields and round numeric columns
        sub_sorted = sub_sorted[[
            'cmdb_id','kpi_name','fault_start_ts','points','max_value','p95','max_severity','min_value','p5','min_severity'
        ]].copy()
        for col in ['max_value','p95','max_severity','min_value','p5','min_severity']:
            sub_sorted[col] = sub_sorted[col].apply(lambda x: round(float(x),6) if pd.notna(x) else None)
        candidate_faults_rows.append(sub_sorted.assign(candidate_cmdb=cmdb))
    # Combine candidate_faults_rows into single DataFrame (if any)
    if candidate_faults_rows:
        candidate_faults_df = pd.concat(candidate_faults_rows, ignore_index=True)
        # reorder columns: candidate_cmdb used to group; keep it
        candidate_faults_df = candidate_faults_df[['candidate_cmdb','cmdb_id','kpi_name','fault_start_ts','points','max_value','p95','max_severity','min_value','p5','min_severity']]
    else:
        candidate_faults_df = pd.DataFrame(columns=['candidate_cmdb','cmdb_id','kpi_name','fault_start_ts','points','max_value','p95','max_severity','min_value','p5','min_severity'])
    candidate_agg_df = pd.DataFrame(candidate_agg_rows, columns=['cmdb_id','highest_max_severity_across_faults','earliest_fault_start_ts','number_of_faults_in_metric_faults'])
    # Round numeric in agg
    candidate_agg_df['highest_max_severity_across_faults'] = candidate_agg_df['highest_max_severity_across_faults'].apply(lambda x: round(float(x),6) if pd.notna(x) else None)
    return downstream_sorted, candidate_agg_df, candidate_faults_df

# Run for both clusters and store kernel variables
trace_downstream_rankings_1647748800, metric_agg_1647748800, metric_faults_top_1647748800 = analyze_cluster_downstream_and_metrics(1647748800, trace_incident_df, metric_faults)
trace_downstream_rankings_1647749520, metric_agg_1647749520, metric_faults_top_1647749520 = analyze_cluster_downstream_and_metrics(1647749520, trace_incident_df, metric_faults)

# For convenience combine metric aggregates and faults into tuples (keeps DataFrame display friendly)
metric_candidates_faults_1647748800 = (metric_agg_1647748800, metric_faults_top_1647748800)
metric_candidates_faults_1647749520 = (metric_agg_1647749520, metric_faults_top_1647749520)

# Expose the compact results (DataFrames). The kernel will display these variables.
trace_downstream_rankings_1647748800, trace_downstream_rankings_1647749520, metric_candidates_faults_1647748800, metric_candidates_faults_1647749520

```
Out[9]:
```
Summary (plain English):

Overview
- We analyzed downstream trace anomalies and corresponding metric faults for two clusters: 1647748800 and 1647749520 (10‑minute windows).
- Output tables are stored in kernel variables:
  - trace_downstream_rankings_1647748800, trace_downstream_rankings_1647749520
  - metric_candidates_faults_1647748800, metric_candidates_faults_1647749520

Key findings — cluster 1647748800
- Top downstream trace contributors (top 20 by incoming anomalous trace groups):
  - productcatalogservice-1 (incoming groups=36, total anomalous points=175)
  - productcatalogservice-2 (36, 162)
  - productcatalogservice-0 (33, 161)
  - cartservice-0/1/2 (24 each; total anomalous points ~223–237)
  - several shippingservice and currencyservice instances also high
  - checkoutservice instances show many outgoing anomalous trace groups (e.g., checkoutservice-0: outgoing=68) and very high total anomalous points (checkoutservice-0 total_anomalous_points=518).
- Metric evidence (top candidate aggregates):
  - productcatalogservice family shows elevated metric severities (highest_max_severity ≈ 1.93, 2.20, 2.47 for different instances).
  - cartservice and cartservice-2 have notable metric severities (cartservice-2 highest_max_severity ≈ 4.12).
  - shippingservice-1 shows a high metric severity ≈ 7.79.
  - currencyservice-2 shows very high metric severity ≈ 12.44.
  - Many candidate cmdb_ids have dozens–>100 metric faults (e.g., productcatalogservice-0 has ~117 metric faults recorded).
- Representative worst KPIs (examples from top candidates):
  - mesh.destination / mesh.source metrics between services (e.g., mesh.destination.recommendationservice…)
  - container/node metrics (container sockets, file descriptors, memory failures)
  - system I/O and network related KPIs appear among highest severities.

Key findings — cluster 1647749520
- Top downstream trace contributors (top 20):
  - productcatalogservice-0/2/1 leading incoming anomalous groups (39, 38, 37 respectively) with high anomalous point totals.
  - cartservice family and currencyservice instances also high.
  - checkoutservice instances again show many outgoing anomalies and very large totals (checkoutservice-0: outgoing roughly 71 and total_anomalous_points ~531).
- Metric evidence (top candidate aggregates):
  - productcatalogservice-0 shows the largest per-candidate metric severity (highest_max_severity ≈ 2.465257).
  - productcatalogservice-2/1 also elevated (≈2.199872, ≈1.927027).
  - some checkoutservice and shippingservice metric faults have high severities (examples in the per-candidate tables show max_severity values up to multiple times p95).
  - Many candidate cmdb_ids again have large numbers of metric faults (tens to >100).

Interpretation and next steps
- Trace anomalies concentrate on incoming traces to the productcatalogservice family, cartservice family, currencyservice, and shippingservice — these are top downstream targets receiving anomalous upstream behavior.
- Checkoutservice instances emit many outgoing anomalous traces (large outgoing counts), suggesting they are a major origin or intermediary in the anomaly propagation.
- Metric evidence aligns with trace findings: productcatalogservice, cartservice, shippingservice, and certain currencyservice nodes show metric faults with high severities (some metrics 2–12× their p95).
- Actionable focus: prioritize investigating productcatalogservice and its containers, checkoutservice interactions, and the shipping/currency nodes which show both many trace anomalies and high metric severities (memory failures, container/network I/O, system I/O). Correlate these candidates with logs and node-level telemetry (and verify p95/p5 threshold validity where severities are NaN) to pinpoint root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  incoming_anomalous_groups  outgoing_anomalous_groups  total_anomalous_points  earliest_anomaly_ts
0   productcatalogservice-1                         36                          0                     175           1647748800
1   productcatalogservice-2                         36                          0                     162           1647748800
2   productcatalogservice-0                         33                          0                     161           1647748800
3             cartservice-0                         24                          0                     237           1647748800
4             cartservice-1                         24                          0                     223           1647748800
5             cartservice-2                         24                          0                     223           1647748800
6         shippingservice-0                         23                          0                     117           1647748800
7         shippingservice-1                         23                          0                     110           1647748800
8         currencyservice-1                         21                          0                     100           1647748800
9         currencyservice-0                         21                          0                      99           1647748800
10        shippingservice-2                         20                          0                     108           1647748800
11        currencyservice-2                         18                          0                      88           1647748800
12        checkoutservice-0                         14                         68                     518           1647748800
13        checkoutservice-2                         13                         68                     512           1647748800
14  recommendationservice-0                         12                         11                     120           1647748800
15         paymentservice-1                         12                          0                      77           1647748800
16              adservice-2                         12                          0                      75           1647748800
17           emailservice-0                         12                          0                      74           1647748800
18           emailservice-2                         12                          0                      74           1647748800
19         paymentservice-2                         12                          0                      70           1647748800,                     cmdb_id  incoming_anomalous_groups  outgoing_anomalous_groups  total_anomalous_points  earliest_anomaly_ts
0   productcatalogservice-0                         39                          0                     181           1647749520
1   productcatalogservice-2                         38                          0                     185           1647749520
2   productcatalogservice-1                         37                          0                     167           1647749520
3             cartservice-0                         24                          0                     231           1647749520
4             cartservice-1                         24                          0                     231           1647749520
5         currencyservice-2                         24                          0                     111           1647749520
6             cartservice-2                         23                          0                     218           1647749520
7         shippingservice-1                         23                          0                     112           1647749520
8         currencyservice-0                         23                          0                      98           1647749520
9         shippingservice-2                         22                          0                     106           1647749520
10        currencyservice-1                         22                          0                      98           1647749520
11        shippingservice-0                         21                          0                     101           1647749520
12        checkoutservice-0                         15                         71                     531           1647749520
13        checkoutservice-2                         15                         69                     481           1647749520
14        checkoutservice-1                         14                         70                     506           1647749520
15  recommendationservice-2                         12                         11                     130           1647749520
16  recommendationservice-1                         12                         12                     127           1647749520
17         paymentservice-0                         12                          0                      82           1647749520
18         paymentservice-2                         12                          0                      75           1647749520
19           emailservice-2                         12                          0                      74           1647749520, (                   cmdb_id  highest_max_severity_across_faults  earliest_fault_start_ts  number_of_faults_in_metric_faults
0  productcatalogservice-1                            1.927027               1647748800                                 94
1  productcatalogservice-2                            2.199872               1647748800                                116
2  productcatalogservice-0                            2.465257               1647748800                                117
3            cartservice-0                            1.047619               1647748800                                 95
4            cartservice-1                            3.826788               1647748800                                 85
5            cartservice-2                            4.118282               1647748800                                101
6        shippingservice-0                            1.342857               1647748800                                 81
7        shippingservice-1                            7.794832               1647748800                                 88
8        currencyservice-1                            2.000000               1647748800                                 66
9        currencyservice-0                            2.298958               1647748800                                 66,              candidate_cmdb                  cmdb_id                                           kpi_name  fault_start_ts  points     max_value           p95  max_severity     min_value            p5  min_severity
0   productcatalogservice-1  productcatalogservice-1  mesh.destination.recommendationservice2.produc...      1647749520       3      0.017825      0.009250      1.927027      0.009250      0.000000      0.000000
1   productcatalogservice-1  productcatalogservice-1  mesh.destination.recommendationservice.product...      1647749520       3      0.066850      0.046825      1.427656      0.046900      0.000000      0.000000
2   productcatalogservice-1  productcatalogservice-1                 container.node-6.container_sockets      1647748800      31      4.500000      4.000000      1.125000      4.000000      4.000000      1.000000
3   productcatalogservice-1  productcatalogservice-1      container.node-6.container_cpu_system_seconds      1647748800       2      0.525000      0.495000      1.060606      0.525000      0.350000      0.666667
4   productcatalogservice-1  productcatalogservice-1        container.node-6.container_file_descriptors      1647748800      31     10.500000     10.000000      1.050000     10.000000     10.000000      1.000000
..                      ...                      ...                                                ...             ...     ...           ...           ...           ...           ...           ...           ...
45        currencyservice-0        currencyservice-0  container.node-6.container_network_receive_MB....      1647749940       2      0.447506      0.194656      2.298958      0.444065      0.120718      0.271848
46        currencyservice-0        currencyservice-0  mesh.destination.checkoutservice.currencyservi...      1647750300       3  20925.000000  19575.000000      1.068966  11475.000000  11475.000000      1.000000
47        currencyservice-0        currencyservice-0  mesh.destination.checkoutservice.currencyservi...      1647750300       2   9957.500000   9342.500000      1.065828   5472.500000   5472.500000      1.000000
48        currencyservice-0        currencyservice-0  mesh.destination.checkoutservice.currencyservi...      1647750300       2     15.500000     15.000000      1.033333      8.500000      8.500000      1.000000
49        currencyservice-0        currencyservice-0  mesh.destination.checkoutservice.currencyservi...      1647750300       2     15.500000     15.000000      1.033333      8.500000      8.500000      1.000000

[50 rows x 11 columns]), (                   cmdb_id  highest_max_severity_across_faults  earliest_fault_start_ts  number_of_faults_in_metric_faults
0  productcatalogservice-0                            2.465257               1647748800                                117
1  productcatalogservice-2                            2.199872               1647748800                                116
2  productcatalogservice-1                            1.927027               1647748800                                 94
3            cartservice-0                            1.047619               1647748800                                 95
4            cartservice-1                            3.826788               1647748800                                 85
5        currencyservice-2                           12.438296               1647748800                                 72
6            cartservice-2                            4.118282               1647748800                                101
7        shippingservice-1                            7.794832               1647748800                                 88
8        currencyservice-0                            2.298958               1647748800                                 66
9        shippingservice-2                            4.666667               1647748800                                 74,              candidate_cmdb                  cmdb_id                                           kpi_name  fault_start_ts  points  max_value        p95  max_severity  min_value       p5  min_severity
0   productcatalogservice-0  productcatalogservice-0  mesh.destination.recommendationservice2.produc...      1647749520       4   0.020400   0.008275      2.465257   0.008275   0.0000      0.000000
1   productcatalogservice-0  productcatalogservice-0  mesh.source.productcatalogservice.jaeger-colle...      1647749520       3   0.025950   0.015675      1.655502   0.015675   0.0042      0.267943
2   productcatalogservice-0  productcatalogservice-0  mesh.destination.recommendationservice.product...      1647749520       3   0.062000   0.039100      1.585678   0.054625   0.0000      0.000000
3   productcatalogservice-0  productcatalogservice-0  mesh.destination.recommendationservice2.produc...      1647750000       2  25.000000  22.500000      1.111111  24.000000  16.5000      0.687500
4   productcatalogservice-0  productcatalogservice-0  mesh.destination.recommendationservice2.produc...      1647750000       2  25.000000  22.500000      1.111111  24.000000  16.5000      0.687500
..                      ...                      ...                                                ...             ...     ...        ...        ...           ...        ...      ...           ...
45        shippingservice-2        shippingservice-2       container.node-5.container_fs_reads./dev/vda      1647748800      31   4.666667   1.000000      4.666667   0.000000   0.0000      0.000000
46        shippingservice-2        shippingservice-2  container.node-5.container_memory_failures.con...      1647748800      31  66.000000  16.500000      4.000000   0.000000   0.0000      0.000000
47        shippingservice-2        shippingservice-2  container.node-5.container_memory_failures.hie...      1647748800      31  66.000000  16.500000      4.000000   0.000000   0.0000      0.000000
48        shippingservice-2        shippingservice-2    container.node-5.container_fs_reads_MB./dev/vda      1647750240       7   0.054688   0.031250      1.750000   0.000000   0.0000      0.000000
49        shippingservice-2        shippingservice-2  mesh.source.shippingservice.jaeger-collector.i...      1647749280      13   0.003050   0.002050      1.487805   0.000000   0.0000      0.000000

[50 rows x 11 columns]))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse kernel variables: trace_incident_df, metric_faults (full faults table)
# Ensure trace_incident_df and metric_faults exist
# Define cluster timestamps
cluster_ts_list = [1647748800, 1647749520]

def extract_service_from_trace_name(trace_name, prefix):
    # prefix is 'trace.from_' or 'trace.to_'
    try:
        s = str(trace_name)
        idx = s.find(prefix)
        if idx == -1:
            return None
        rest = s[idx + len(prefix):]
        # service name is up to next '.' or end
        m = re.match(r'([^\.]+)', rest)
        return m.group(1) if m else None
    except Exception:
        return None

def analyze_sinks_for_cluster(cluster_ts):
    start = int(cluster_ts)
    end = int(cluster_ts + 600)
    # Filter trace rows in window
    win = trace_incident_df[(trace_incident_df['timestamp'] >= start) & (trace_incident_df['timestamp'] <= end)].copy()
    if win.empty:
        return pd.DataFrame(columns=['cmdb_id','incoming_anomalous_groups','outgoing_anomalous_groups','sink_score','total_anomalous_points','earliest_anomaly_ts']), pd.DataFrame(columns=['cmdb_id','highest_max_severity_across_faults','earliest_fault_start_ts','number_of_metric_faults','top_kpis'])
    # Ensure numeric and thresholds present
    win['value'] = pd.to_numeric(win['value'], errors='coerce')
    win['p95'] = pd.to_numeric(win.get('p95'), errors='coerce')
    win['p5'] = pd.to_numeric(win.get('p5'), errors='coerce')
    # Mark anomalous rows
    win['is_high'] = (~win['p95'].isna()) & (win['value'] >= win['p95'])
    win['is_low'] = (~win['p5'].isna()) & (win['value'] <= win['p5'])
    win['is_anom'] = win['is_high'] | win['is_low']
    anoms = win[win['is_anom']].copy()
    if anoms.empty:
        return pd.DataFrame(columns=['cmdb_id','incoming_anomalous_groups','outgoing_anomalous_groups','sink_score','total_anomalous_points','earliest_anomaly_ts']), pd.DataFrame(columns=['cmdb_id','highest_max_severity_across_faults','earliest_fault_start_ts','number_of_metric_faults','top_kpis'])
    # Extract incoming/outgoing service names per anomalous trace row
    anoms['incoming_service'] = anoms['trace_name'].apply(lambda x: extract_service_from_trace_name(x, 'trace.from_'))
    anoms['outgoing_service'] = anoms['trace_name'].apply(lambda x: extract_service_from_trace_name(x, 'trace.to_'))
    # Only consider rows where extraction succeeded for incoming/outgoing groups separately
    # Count distinct incoming/outgoing trace_name groups per cmdb_id
    incoming_groups = anoms[anoms['incoming_service'].notna()].groupby('cmdb_id')['trace_name'].nunique().rename('incoming_anomalous_groups')
    outgoing_groups = anoms[anoms['outgoing_service'].notna()].groupby('cmdb_id')['trace_name'].nunique().rename('outgoing_anomalous_groups')
    total_points = anoms.groupby('cmdb_id').size().rename('total_anomalous_points')
    earliest = anoms.groupby('cmdb_id')['timestamp'].min().rename('earliest_anomaly_ts')
    sinks = pd.concat([incoming_groups, outgoing_groups, total_points, earliest], axis=1).fillna(0)
    sinks['incoming_anomalous_groups'] = sinks['incoming_anomalous_groups'].astype(int)
    sinks['outgoing_anomalous_groups'] = sinks['outgoing_anomalous_groups'].astype(int)
    sinks['total_anomalous_points'] = sinks['total_anomalous_points'].astype(int)
    sinks['earliest_anomaly_ts'] = sinks['earliest_anomaly_ts'].apply(lambda x: int(x) if pd.notna(x) and x!=0 else None)
    sinks = sinks.reset_index()
    sinks['sink_score'] = sinks['incoming_anomalous_groups'] - sinks['outgoing_anomalous_groups']
    # Sort by sink_score DESC then incoming_anomalous_groups DESC
    sinks_sorted = sinks.sort_values(['sink_score','incoming_anomalous_groups'], ascending=[False, False]).head(20).reset_index(drop=True)
    # Step 5: For top 10 sinks, get metric evidence from metric_faults
    top10 = sinks_sorted['cmdb_id'].head(10).tolist()
    metric_rows = []
    for cmdb in top10:
        sub = metric_faults[metric_faults['cmdb_id'] == cmdb].copy()
        if sub.empty:
            metric_rows.append({
                'cmdb_id': cmdb,
                'highest_max_severity_across_faults': None,
                'earliest_fault_start_ts': None,
                'number_of_metric_faults': 0,
                'top_kpis': []
            })
            continue
        # ensure numeric severities
        sub['max_severity'] = pd.to_numeric(sub.get('max_severity'), errors='coerce')
        # aggregate
        highest = sub['max_severity'].dropna()
        highest_val = float(highest.max()) if highest.shape[0]>0 else None
        earliest_fault = int(sub['fault_start_ts'].min())
        num_faults = int(sub.shape[0])
        # top 3 KPIs by max_severity
        sub['_sort_sev'] = sub['max_severity'].fillna(-np.inf)
        topk = sub.sort_values(['_sort_sev'], ascending=[False]).head(3)[['kpi_name','max_severity']].to_dict(orient='records')
        # format severities to 6 decimals or None
        formatted_topk = []
        for rec in topk:
            sev = rec.get('max_severity')
            sev = round(float(sev),6) if pd.notna(sev) else None
            formatted_topk.append({'kpi_name': str(rec.get('kpi_name')), 'max_severity': sev})
        metric_rows.append({
            'cmdb_id': cmdb,
            'highest_max_severity_across_faults': round(highest_val,6) if highest_val is not None else None,
            'earliest_fault_start_ts': earliest_fault,
            'number_of_metric_faults': num_faults,
            'top_kpis': formatted_topk
        })
    metric_evidence_df = pd.DataFrame(metric_rows, columns=['cmdb_id','highest_max_severity_across_faults','earliest_fault_start_ts','number_of_metric_faults','top_kpis'])
    return sinks_sorted[[
        'cmdb_id','incoming_anomalous_groups','outgoing_anomalous_groups','sink_score','total_anomalous_points','earliest_anomaly_ts'
    ]], metric_evidence_df

# Run for both clusters and store results in kernel variables
sink_ranking_1647748800, sink_metric_evidence_1647748800 = analyze_sinks_for_cluster(1647748800)
sink_ranking_1647749520, sink_metric_evidence_1647749520 = analyze_sinks_for_cluster(1647749520)

# Display compact results (DataFrames)
sink_ranking_1647748800.head(20), sink_metric_evidence_1647748800.head(10), sink_ranking_1647749520.head(20), sink_metric_evidence_1647749520.head(10)

```
Out[10]:
```
Summary of sink analysis (plain English):

- Method: we parsed anomalous traces in the 10-minute windows after each cluster start and built incoming/outgoing edges from trace.from_/trace.to_ names. We ranked sinks by sink_score = incoming_anomalous_groups − outgoing_anomalous_groups and then examined metric fault evidence for the top sinks. Results are stored in kernel variables:
  - sink_ranking_1647748800, sink_ranking_1647749520
  - sink_metric_evidence_1647748800, sink_metric_evidence_1647749520
  - metric_faults contains the full metric fault table used for evidence.

Key results for cluster 1647748800 (window 1647748800–1647749400)
- Top sinks (by sink_score): productcatalogservice-1, productcatalogservice-2, productcatalogservice-0 (sink_scores 36,36,33 respectively). Other top sinks include cartservice-0/1/2 (sink_score 24), shippingservice-0/1/2, and currencyservice instances.
- Counts: productcatalogservice-1 had incoming_anomalous_groups=36 and total_anomalous_points=175; cartservice-0 had incoming_anomalous_groups=24 and total_anomalous_points=237.
- Metric evidence for top sinks:
  - productcatalogservice-0/1/2 show elevated metric severities (highest_max_severity ≈ 2.465257, 1.927027, 2.199872 respectively) with many metric faults (e.g., 94–117 faults).
  - cartservice-2 highest_max_severity ≈ 4.118282; cartservice-1 ≈ 3.826788.
  - shippingservice-1 highest_max_severity ≈ 7.794832.
  - currencyservice-2 highest_max_severity ≈ 12.438296 (very high).
- Top metric KPIs driving severity are mesh (mesh.source/mesh.destination) metrics between services and node/container metrics (network, file descriptors, memory failures, I/O).

Key results for cluster 1647749520 (window 1647749520–164774...+600)
- Top sinks: productcatalogservice-0, productcatalogservice-2, productcatalogservice-1 (sink_scores 39,38,37). Cartservice and currencyservice families again rank high.
- Metric evidence:
  - productcatalogservice-0 shows the largest per-candidate metric severity (highest_max_severity ≈ 2.465257), with many faults (~117).
  - Other productcatalogservice instances also show elevated severities (≈2.20, ≈1.93).
  - Several cartservice and shippingservice faults remain high (examples with max_severity > 4).
- Checkoutservice instances appear as high-volume trace emitters (many outgoing anomalous groups) and overall are heavily involved in anomaly propagation.

Interpretation and actionable pointers
- The trace-derived sink ranking consistently highlights the productcatalogservice family, cartservice family, shippingservice, and currencyservice as major downstream receivers of anomalous traffic.
- Metric evidence corroborates these sinks: many high-severity metric faults appear on those services (mesh metrics, container/network, memory failures, and node I/O), with some metrics showing multi‑x p95 amplifications (examples: currencyservice-2 ~12.4×, shippingservice-1 ~7.8×, cartservice-2 ~4.1×).
- Checkoutservice components show heavy outgoing anomalous trace activity and should be investigated as likely upstream contributors or intermediaries.
- Next steps: prioritize inspecting productcatalogservice, cartservice, currencyservice, shippingservice, and checkoutservice interactions. Focus on mesh communication metrics, container memory failures, and node I/O/network KPIs. Also verify threshold definitions (p95/p5) for any NaN/severity issues when p95==0.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  incoming_anomalous_groups  outgoing_anomalous_groups  sink_score  total_anomalous_points  earliest_anomaly_ts
0   productcatalogservice-1                         36                          0          36                     175           1647748800
1   productcatalogservice-2                         36                          0          36                     162           1647748800
2   productcatalogservice-0                         33                          0          33                     161           1647748800
3             cartservice-0                         24                          0          24                     237           1647748800
4             cartservice-1                         24                          0          24                     223           1647748800
5             cartservice-2                         24                          0          24                     223           1647748800
6         shippingservice-0                         23                          0          23                     117           1647748800
7         shippingservice-1                         23                          0          23                     110           1647748800
8         currencyservice-0                         21                          0          21                      99           1647748800
9         currencyservice-1                         21                          0          21                     100           1647748800
10        shippingservice-2                         20                          0          20                     108           1647748800
11        currencyservice-2                         18                          0          18                      88           1647748800
12              adservice-0                         12                          0          12                      63           1647748800
13              adservice-1                         12                          0          12                      64           1647748800
14              adservice-2                         12                          0          12                      75           1647748800
15           emailservice-0                         12                          0          12                      74           1647748800
16           emailservice-2                         12                          0          12                      74           1647748800
17         paymentservice-1                         12                          0          12                      77           1647748800
18         paymentservice-2                         12                          0          12                      70           1647748800
19           emailservice-1                         11                          0          11                      72           1647748800,                    cmdb_id  highest_max_severity_across_faults  earliest_fault_start_ts  number_of_metric_faults                                           top_kpis
0  productcatalogservice-1                            1.927027               1647748800                       94  [{'kpi_name': 'mesh.destination.recommendation...
1  productcatalogservice-2                            2.199872               1647748800                      116  [{'kpi_name': 'mesh.destination.recommendation...
2  productcatalogservice-0                            2.465257               1647748800                      117  [{'kpi_name': 'mesh.destination.recommendation...
3            cartservice-0                            1.047619               1647748800                       95  [{'kpi_name': 'mesh.destination.checkoutservic...
4            cartservice-1                            3.826788               1647748800                       85  [{'kpi_name': 'container.node-6.container_netw...
5            cartservice-2                            4.118282               1647748800                      101  [{'kpi_name': 'container.node-6.container_netw...
6        shippingservice-0                            1.342857               1647748800                       81  [{'kpi_name': 'mesh.destination.checkoutservic...
7        shippingservice-1                            7.794832               1647748800                       88  [{'kpi_name': 'container.node-6.container_netw...
8        currencyservice-0                            2.298958               1647748800                       66  [{'kpi_name': 'container.node-6.container_netw...
9        currencyservice-1                            2.000000               1647748800                       66  [{'kpi_name': 'container.node-6.container_cpu_...,                     cmdb_id  incoming_anomalous_groups  outgoing_anomalous_groups  sink_score  total_anomalous_points  earliest_anomaly_ts
0   productcatalogservice-0                         39                          0          39                     181           1647749520
1   productcatalogservice-2                         38                          0          38                     185           1647749520
2   productcatalogservice-1                         37                          0          37                     167           1647749520
3             cartservice-0                         24                          0          24                     231           1647749520
4             cartservice-1                         24                          0          24                     231           1647749520
5         currencyservice-2                         24                          0          24                     111           1647749520
6             cartservice-2                         23                          0          23                     218           1647749520
7         currencyservice-0                         23                          0          23                      98           1647749520
8         shippingservice-1                         23                          0          23                     112           1647749520
9         currencyservice-1                         22                          0          22                      98           1647749520
10        shippingservice-2                         22                          0          22                     106           1647749520
11        shippingservice-0                         21                          0          21                     101           1647749520
12              adservice-1                         12                          0          12                      61           1647749520
13              adservice-2                         12                          0          12                      61           1647749520
14           emailservice-0                         12                          0          12                      72           1647749520
15           emailservice-1                         12                          0          12                      73           1647749520
16           emailservice-2                         12                          0          12                      74           1647749520
17         paymentservice-0                         12                          0          12                      82           1647749520
18         paymentservice-2                         12                          0          12                      75           1647749520
19              adservice-0                         11                          0          11                      59           1647749520,                    cmdb_id  highest_max_severity_across_faults  earliest_fault_start_ts  number_of_metric_faults                                           top_kpis
0  productcatalogservice-0                            2.465257               1647748800                      117  [{'kpi_name': 'mesh.destination.recommendation...
1  productcatalogservice-2                            2.199872               1647748800                      116  [{'kpi_name': 'mesh.destination.recommendation...
2  productcatalogservice-1                            1.927027               1647748800                       94  [{'kpi_name': 'mesh.destination.recommendation...
3            cartservice-0                            1.047619               1647748800                       95  [{'kpi_name': 'mesh.destination.checkoutservic...
4            cartservice-1                            3.826788               1647748800                       85  [{'kpi_name': 'container.node-6.container_netw...
5        currencyservice-2                           12.438296               1647748800                       72  [{'kpi_name': 'container.node-6.container_cpu_...
6            cartservice-2                            4.118282               1647748800                      101  [{'kpi_name': 'container.node-6.container_netw...
7        currencyservice-0                            2.298958               1647748800                       66  [{'kpi_name': 'container.node-6.container_netw...
8        shippingservice-1                            7.794832               1647748800                       88  [{'kpi_name': 'container.node-6.container_netw...
9        currencyservice-1                            2.000000               1647748800                       66  [{'kpi_name': 'container.node-6.container_cpu_...)```
```